# Classificação de Notícias Econômicas com LLMs

## Projeto IBRE/FGV — Índice de Incerteza Econômica


### Contexto do Projeto

O **IBRE (Instituto Brasileiro de Economia)**, vinculado à **Fundação Getulio Vargas (FGV)**, é uma das principais instituições de pesquisa econômica do Brasil. Entre seus diversos indicadores, o IBRE produz o **Indicador de Incerteza da Economia (IIE-Br)**, que mede o grau de incerteza econômica percebido no país.

Um dos componentes desse indicador é construído a partir da **análise de notícias publicadas na mídia brasileira**. A ideia central é simples e poderosa: quando há mais incerteza na economia, os jornais tendem a publicar mais matérias que mencionam termos ligados à incerteza. Ao quantificar sistematicamente essas menções, conseguimos construir uma medida objetiva do "humor" econômico do país.

Tradicionalmente, essa classificação era feita com abordagens baseadas em **palavras-chave e regras manuais**. Com o avanço dos **Modelos de Linguagem de Grande Escala (LLMs)**, surge a oportunidade de realizar essa tarefa de forma mais precisa, capturando nuances semânticas que regras simples não conseguem detectar.


### O que este notebook faz

Este notebook implementa um **pipeline de classificação automática de notícias econômicas brasileiras** utilizando LLMs. Dado o texto de uma notícia, o modelo classifica o artigo em **quatro dimensões**:

| # | Dimensão | Pergunta que responde | Valores possíveis |
|---|----------|----------------------|--------------------|
| 1 | **Escopo geográfico** | A notícia trata de um tema internacional ou nacional? | `internacional` · `nacional` |
| 2 | **Incerteza econômica** | A notícia menciona ou discute incerteza econômica? | `sim` · `não` |
| 3 | **Incerteza afeta o Brasil** | A incerteza mencionada tem impacto (direto ou indireto) sobre a economia brasileira? | `sim` · `não` |
| 4 | **Polaridade (sentimento)** | Qual é o tom predominante da notícia em relação à economia? | `positivo` · `negativo` · `neutro` |

Essas quatro dimensões, combinadas, permitem construir uma visão rica sobre como a imprensa está retratando o cenário econômico em um dado momento.


### Por que isso importa?

A **incerteza econômica** é um dos fatores mais relevantes para decisões de investimento, consumo e política monetária. Quando a incerteza é alta:

- Empresas **adiam investimentos** e contratações
- Consumidores **reduzem gastos** e aumentam poupança precaucional
- O **Banco Central** precisa ponderar a incerteza ao definir a taxa de juros
- Mercados financeiros apresentam **maior volatilidade**

Medir essa incerteza de forma sistemática e em tempo real — usando milhares de notícias publicadas diariamente — é um desafio que se beneficia enormemente da capacidade dos LLMs de compreender contexto e nuance em texto.

Ao automatizar a classificação com LLMs, ganhamos:

- **Escala**: capacidade de processar grandes volumes de notícias diariamente
- **Consistência**: redução da variabilidade inerente à classificação humana
- **Velocidade**: resultados quase em tempo real
- **Reprodutibilidade**: o mesmo prompt aplicado ao mesmo texto gera resultados comparáveis


### Estrutura técnica do notebook

Este notebook contém **três abordagens** em diferentes estágios de maturidade:

| Abordagem | API / Modelo | Status | Descrição |
|-----------|-------------|--------|-----------|
| **Exemplo principal** | OpenAI Chat Completions (`gpt-3.5-turbo`) | ✅ Testado e funcional | Pipeline completo com prompts validados, classificação em lote e métricas de avaliação. Este é o ponto de partida recomendado. |
| **Framework WIP 1** | OpenAI Responses API (modelos modernos) | ⚠️ Em desenvolvimento | Utiliza a API mais recente da OpenAI com saída estruturada nativa (strict JSON). |
| **Framework WIP 2** | Google Gemini | ⚠️ Em desenvolvimento | Integração com os modelos Gemini do Google — alternativa gratuita ao ecossistema OpenAI. |

> **WIP** = *Work in Progress* (trabalho em andamento). Os frameworks WIP foram montados mas **não foram testados**. Eles estão incluídos para que voluntários possam contribuir com seu desenvolvimento, teste e depuração. Se encontrar erros, consulte os links de documentação fornecidos em cada seção.

Recomendamos que você **comece pelo exemplo principal** (`gpt-3.5-turbo`), entenda o pipeline de ponta a ponta, e depois explore os frameworks em desenvolvimento como oportunidade de contribuição.


---

## Objetivos de Aprendizado


Ao concluir este notebook, você será capaz de:

### 1. Engenharia de Prompts (*Prompt Engineering*)

- **Projetar prompts eficazes** para tarefas de classificação de texto com LLMs
- Entender a diferença entre os papéis `system`, `user` e `assistant` em APIs de chat
- Aplicar técnicas como **few-shot prompting** (fornecer exemplos no prompt) para melhorar a qualidade das respostas
- Iterar sobre prompts de forma sistemática, avaliando o impacto de cada mudança
- Estruturar instruções para que o modelo retorne **respostas em formato padronizado** (ex.: JSON), facilitando o pós-processamento

### 2. Uso de APIs de LLMs

- **Configurar e autenticar** chamadas às APIs da OpenAI e do Google Gemini
- Compreender os **parâmetros principais** de uma chamada de API (modelo, temperatura, tokens máximos, etc.)
- Implementar **chamadas em lote** para processar múltiplas notícias de forma eficiente
- Tratar erros comuns: limites de taxa (*rate limits*), timeouts, respostas malformadas
- Estimar e monitorar **custos** de uso da API com base no número de tokens consumidos

### 3. Métricas de Avaliação

- Calcular métricas clássicas de classificação: **acurácia, precisão, recall e F1-score**
- Interpretar uma **matriz de confusão** para identificar padrões de erro do modelo
- Comparar o desempenho do LLM com classificações humanas de referência (*ground truth*)
- Avaliar criticamente quando as métricas indicam que o prompt precisa ser ajustado

### 4. Habilidades Práticas de Projeto

- Trabalhar com **DataFrames do pandas** para organizar entradas e resultados
- Fazer **parse de respostas JSON** retornadas por LLMs
- Construir um pipeline reprodutível que vai da notícia bruta até a classificação final
- Documentar decisões e resultados de forma clara para colaboração em equipe


## Estrutura do Notebook

O notebook está organizado nas seguintes seções:

| Seção | Conteúdo | O que você vai fazer |
|-------|----------|---------------------|
| **1. Introdução** *(você está aqui)* | Contexto do projeto, objetivos e estrutura | Ler e entender o escopo do trabalho |
| **2. Configuração do Ambiente** | Instalação de pacotes, importações e configuração de chaves de API | Executar as células para preparar o ambiente |
| **3. Carregamento dos Dados** | Leitura do dataset de notícias e exploração inicial | Inspecionar os dados e entender o formato |
| **4. Classificação com GPT-3.5-Turbo** | Pipeline completo: prompt → classificação → parsing com regex | Estudar o exemplo, entender cada decisão |
| **5. Avaliação de Resultados** | Cálculo de métricas, matrizes de confusão e análise de erros | Interpretar os resultados e identificar pontos de melhoria |
| **6. Agora É Sua Vez** | Espaço para iterar seus próprios prompts e comparar resultados | Experimentar variações e documentar suas descobertas |
| **7. Framework WIP — OpenAI Moderna** | Estrutura para usar a Responses API com strict JSON | Explorar, completar e testar (desafio!) |
| **8. Framework WIP — Google Gemini** | Estrutura para usar modelos Gemini | Explorar, completar e testar (desafio!) |
| **9. Desafios** | Desafios abertos para voluntários | Escolher um desafio e contribuir |
| **10. Recursos** | Links para documentação e referências | Consultar quando precisar de ajuda |


## Como usar este notebook

### Fluxo de trabalho recomendado

1. **Leia e execute o exemplo principal** (seções 1–5)
   - Percorra as células na ordem, lendo as explicações e executando o código
   - Não tenha pressa — entender *por que* cada decisão foi tomada é tão importante quanto ver o código funcionar
   - Preste atenção especial aos **prompts**: eles são o coração deste projeto

2. **Experimente e itere** (seção 6)
   - Modifique os prompts e observe como as classificações mudam
   - Compare suas métricas com o baseline (o exemplo original)
   - Documente o que tentou e por quê — isso é tão importante quanto os resultados

3. **Contribua com os frameworks WIP** (seções 7–9)
   - Depois de dominar o pipeline principal, escolha um dos desafios
   - Complete as implementações, teste com o mesmo dataset e compare resultados
   - Documente suas descobertas para o restante da equipe

### Pré-requisitos

**Necessário:**
- Python básico/intermediário (variáveis, funções, loops, dicionários)
- Familiaridade com pandas (DataFrames, seleção de colunas)
- Saber executar células em Jupyter/Colab

**Não necessário** (será ensinado aqui):
- Modelos de linguagem ou IA
- APIs REST
- Processamento de linguagem natural (NLP)

---

**Vamos começar!** Na próxima seção, configuraremos o ambiente de desenvolvimento.


---

# Configuração do Ambiente


## 1. Instalação de Pacotes

Precisamos instalar algumas bibliotecas Python que usaremos ao longo do projeto.

| Pacote | Para que serve |
|--------|---------------|
| **openai** | Cliente oficial da API da OpenAI — para acessar modelos como o GPT-3.5-Turbo e mais recentes. |
| **google-genai** | Cliente oficial da API do Google Gemini — alternativa gratuita para classificação de texto. |
| **pandas** | Manipulação e análise de dados tabulares (DataFrames). |
| **scikit-learn** | Ferramentas para métricas de avaliação (acurácia, precisão, matriz de confusão, etc.). |
| **openpyxl** | Motor de leitura/escrita de arquivos Excel (.xlsx). O pandas usa por trás dos panos. |

> **Nota:** O comando `pip install` abaixo usa o prefixo `!` porque estamos executando um comando do terminal dentro do Jupyter/Colab. O flag `-q` reduz o texto exibido durante a instalação.


In [ ]:
# Instalação dos pacotes necessários
# -q = modo silencioso (menos texto na tela)
%pip install -q openai google-genai pandas scikit-learn openpyxl python-dotenv

In [ ]:
# Verificação rápida: confirmar que tudo foi instalado corretamente
import importlib

pacotes_para_verificar = {
    "openai": "openai",
    "google-genai": "google.genai",
    "pandas": "pandas",
    "scikit-learn": "sklearn",
    "openpyxl": "openpyxl",
}

print("Verificando instalação dos pacotes...")
print("-" * 40)

todos_ok = True
for nome_pacote, nome_import in pacotes_para_verificar.items():
    try:
        modulo = importlib.import_module(nome_import)
        versao = getattr(modulo, "__version__", "versão não disponível")
        print(f"  ✅ {nome_pacote} ({versao})")
    except ImportError:
        print(f"  ❌ {nome_pacote} — NÃO encontrado! Execute a célula de instalação acima.")
        todos_ok = False

print("-" * 40)
if todos_ok:
    print("Tudo certo! Todos os pacotes estão instalados.")
else:
    print("ATENÇÃO: Alguns pacotes não foram instalados. Releia as mensagens acima.")


## 2. Detecção de Ambiente (Colab vs Local)

Este notebook pode ser executado de duas formas:

1. **Google Colab** — direto no navegador, sem instalar nada no computador.
2. **Jupyter local** — no seu próprio computador (via Jupyter Notebook, JupyterLab ou VS Code).

Algumas funcionalidades (como o gerenciamento seguro de chaves de API) funcionam de forma diferente em cada ambiente. Vamos detectar automaticamente onde estamos rodando.

**Como funciona?** O Google Colab inclui um módulo especial chamado `google.colab` que só existe nesse ambiente. Se conseguirmos importá-lo, sabemos que estamos no Colab.


In [5]:
# Detecção automática do ambiente de execução

try:
    import google.colab  # Este módulo só existe no Google Colab
    AMBIENTE = "colab"
except ImportError:
    AMBIENTE = "local"

print(f"Ambiente detectado: {AMBIENTE.upper()}")

if AMBIENTE == "colab":
    print("Você está rodando no Google Colab.")
    print("As chaves de API serão carregadas via Colab Secrets (mais seguro).")
else:
    print("Você está rodando em um Jupyter local.")
    print("As chaves de API serão carregadas via variáveis de ambiente.")
    print("(Caso não tenha configurado, será solicitada entrada manual.)")


Ambiente detectado: COLAB
Você está rodando no Google Colab.
As chaves de API serão carregadas via Colab Secrets (mais seguro).


## 3. Configurando suas Chaves de API

Para usar os modelos de IA (OpenAI e Google Gemini), precisamos de **chaves de API** — são como senhas que identificam quem está fazendo a requisição.

> **Sobre as chaves deste projeto:**
> - **OPENAI_API_KEY**: você recebeu a sua do(a) instrutor(a). Não precisa criar conta na OpenAI.
> - **GOOGLE_API_KEY**: é sua chave pessoal e **gratuita** do Google AI Studio. Só vai precisar dela na seção 8 (Gemini) — pode configurar depois.

---

### Por que NÃO colocar chaves direto no código?

```python
# ❌ PERIGOSO!
chave = "sk-abc123suachavesecretaaqui"
```

Se você compartilhar o notebook, qualquer pessoa terá acesso à sua chave e poderá gerar custos na sua conta. Em vez disso, vamos usar um **arquivo separado** que fica só no seu computador.

---

### Opção A — Arquivo `.env` (para quem usa VS Code, Cursor ou Jupyter local)

Esta é a opção mais simples para quem roda no próprio computador. Vamos criar um arquivo de texto com suas chaves.

#### Passo 1: Abrir o terminal no seu editor

O terminal é uma janelinha de texto dentro do próprio editor onde você digita comandos. Para abrir:

| Editor | Como abrir o terminal |
|--------|----------------------|
| **VS Code** | Pressione **Ctrl + `** (a crase, tecla acima do Tab) — ou vá no menu **Terminal → Novo Terminal** |
| **Cursor** | Pressione **Ctrl + `** — ou vá no menu **Terminal → Novo Terminal** |
| **JupyterLab** | Clique em **File → New → Terminal** |

Quando abrir, você verá algo assim:

```
C:\Users\SeuNome\MinhaPasta>
```
ou no Mac/Linux:
```
~/MinhaPasta$
```

#### Passo 2: Navegar até a pasta do projeto

No terminal, vá até a pasta onde estão o notebook e o Excel. Copie e cole o comando abaixo, **trocando o caminho** pela pasta real:

**Windows:**
```
cd "C:\Users\SeuNome\Documentos\IBRE"
```

**Mac/Linux:**
```
cd ~/Documentos/IBRE
```

> **Dica:** Se você não sabe o caminho, clique com o botão direito na pasta no explorador de arquivos e procure "Copiar caminho" ou "Copy Path".

#### Passo 3: Criar o arquivo `.env`

Copie e cole **exatamente este comando** no terminal:

**Windows:**
```
echo OPENAI_API_KEY=COLE_SUA_CHAVE_OPENAI_AQUI > .env
```

**Mac/Linux:**
```
echo "OPENAI_API_KEY=COLE_SUA_CHAVE_OPENAI_AQUI" > .env
```

**Agora troque `COLE_SUA_CHAVE_OPENAI_AQUI` pela chave real** que você recebeu do(a) instrutor(a). Exemplo:

```
echo OPENAI_API_KEY=sk-proj-abc123def456ghi789 > .env
```

> **Importante:** A chave não tem espaços e não tem aspas ao redor. Cole exatamente como recebeu.

#### Passo 4: Verificar que funcionou

No terminal, rode:

**Windows:**
```
type .env
```

**Mac/Linux:**
```
cat .env
```

Você deve ver algo assim:
```
OPENAI_API_KEY=sk-proj-abc123def456ghi789
```

Se aparecer, está pronto! O notebook vai encontrar a chave automaticamente.

#### Passo 5 (opcional): Adicionar a chave do Google depois

Quando chegar na seção 8 (Gemini), abra o arquivo `.env` no editor e adicione uma segunda linha:

```
OPENAI_API_KEY=sk-proj-abc123def456ghi789
GOOGLE_API_KEY=AIzaSyABC123suachavegoogleaqui
```

Sua pasta final deve ficar assim:
```
📁 Pasta do projeto/
├── Classificacao_Noticias_IBRE_Educacional.ipynb  ← este notebook
├── Historias Para classificação.xlsx               ← dataset
└── .env                                            ← suas chaves (NÃO compartilhe este arquivo!)
```

---

### Opção B — Google Colab Secrets (para quem usa o Google Colab)

Se você está usando o **Google Colab** (no navegador), siga estes passos:

1. No Colab, clique no ícone de **chave 🔑** na barra lateral esquerda (ou vá em **Configurações → Secrets**).
2. Clique em **"Adicionar novo secret"** (ou "Add new secret").
3. No campo **Nome**, digite exatamente: `OPENAI_API_KEY`
4. No campo **Valor**, cole a chave que você recebeu do(a) instrutor(a).
5. Certifique-se de que a opção **"Acesso ao notebook"** está **ativada** (toggle ligado).
6. Quando chegar na seção 8 (Gemini), repita os passos 2–5 para `GOOGLE_API_KEY`.

**Vantagens do Colab Secrets:**
- A chave fica na sua conta Google, não no notebook.
- Se você compartilhar o notebook, a chave **não** vai junto.

---

### Opção C — Colar na hora (fallback)

Se nenhuma das opções acima funcionar, o notebook vai pedir que você **cole a chave manualmente** quando executar a célula de setup. Funciona, mas você terá que colar novamente toda vez que reiniciar o notebook.

## 4. Função de Carregamento de Chaves de API

Agora vamos criar uma função que carrega as chaves de API de forma inteligente, tentando os métodos disponíveis na seguinte ordem:

1. **Arquivo `.env`** (lido automaticamente por `python-dotenv`)
2. **Colab Secrets** (se estiver no Google Colab)
3. **Variáveis de ambiente** (se configuradas no sistema)
4. **Entrada manual** (fallback — pede para o usuário colar a chave)

Dessa forma, o mesmo notebook funciona em qualquer ambiente sem precisar alterar o código.

In [ ]:
import os
import getpass

# python-dotenv: carrega automaticamente o arquivo .env da pasta atual
# (se o arquivo existir, as chaves ficam disponíveis via os.environ)
try:
    from dotenv import load_dotenv
    load_dotenv()  # Lê o .env e injeta as variáveis em os.environ
    print("python-dotenv: arquivo .env carregado com sucesso." if os.path.exists(".env") else "python-dotenv: nenhum arquivo .env encontrado (ok, tentaremos outros métodos).")
except ImportError:
    print("python-dotenv não instalado — pulando leitura de .env.")
    print("Dica: execute '!pip install python-dotenv' e reinicie o kernel.")


def carregar_chave_api(nome_servico: str) -> str:
    """
    Carrega uma chave de API de forma segura.

    Ordem de tentativa:
        1. Arquivo .env (já carregado por load_dotenv no início)
        2. Google Colab Secrets (se estiver no Colab)
        3. Variável de ambiente do sistema
        4. Entrada manual via getpass (fallback)

    Parâmetros:
        nome_servico (str): Nome da chave, ex: "OPENAI_API_KEY" ou "GOOGLE_API_KEY"

    Retorna:
        str: A chave de API carregada
    """
    chave = None
    origem = None

    # --- Tentativa 1: .env (já foi carregado por load_dotenv acima) ---
    # --- Tentativa 2: Colab Secrets ---
    # Ambas as tentativas ficam disponíveis via os.environ após load_dotenv,
    # então primeiro tentamos Colab Secrets (que é específico) e depois env vars.
    if AMBIENTE == "colab":
        try:
            from google.colab import userdata
            chave = userdata.get(nome_servico)
            origem = "Colab Secrets"
        except Exception as erro:
            print(f"  Colab Secrets: '{nome_servico}' não encontrada. ({type(erro).__name__})")
            print(f"  Dica: Vá em 🔑 Secrets (barra lateral) → Adicionar → Nome: {nome_servico}")

    # --- Tentativa 3: Variável de ambiente (inclui .env carregado por dotenv) ---
    if chave is None:
        chave = os.environ.get(nome_servico)
        if chave is not None:
            origem = "arquivo .env" if os.path.exists(".env") else "variável de ambiente"
        else:
            print(f"  Variável de ambiente: '{nome_servico}' não encontrada.")

    # --- Tentativa 4: Entrada manual (fallback) ---
    if chave is None:
        print()
        print(f"  Nenhuma configuração automática encontrada para '{nome_servico}'.")
        print(f"  Por favor, cole sua chave abaixo.")
        if nome_servico == "OPENAI_API_KEY":
            print(f"  (Esta chave foi fornecida pelo(a) instrutor(a).)")
        elif nome_servico == "GOOGLE_API_KEY":
            print(f"  (Crie sua chave gratuita em: https://aistudio.google.com/apikey)")
        print()
        chave = getpass.getpass(f"  Cole aqui a {nome_servico}: ")
        origem = "entrada manual"

    # --- Validação básica ---
    if not chave or chave.strip() == "":
        print()
        print(f"  ERRO: A chave '{nome_servico}' está vazia!")
        print(f"  Sem essa chave, não será possível usar o serviço correspondente.")
        print(f"  Releia a seção 3 acima para instruções de configuração.")
        raise ValueError(f"Chave de API '{nome_servico}' não foi fornecida ou está vazia.")

    chave = chave.strip()
    print(f"  ✅ '{nome_servico}' carregada com sucesso (via {origem}).")
    return chave


def mascarar_chave(chave: str, caracteres_visiveis: int = 4) -> str:
    """
    Retorna uma versão mascarada da chave, mostrando apenas os primeiros caracteres.
    Exemplo: "sk-abc123xyz" → "sk-a..."
    """
    if len(chave) <= caracteres_visiveis:
        return "****"
    return chave[:caracteres_visiveis] + "..."


print("Funções 'carregar_chave_api' e 'mascarar_chave' definidas com sucesso.")

### Carregando as chaves

Agora vamos usar a função para carregar as duas chaves de API.

Se tudo estiver configurado corretamente (Colab Secrets ou variáveis de ambiente), as chaves serão carregadas automaticamente. Caso contrário, o notebook pedirá que você cole cada chave manualmente.


In [ ]:
# Carregamento da chave da OpenAI
# (fornecida pelo(a) instrutor(a) — cada aluno recebe a sua)
print("Carregando OPENAI_API_KEY...")
OPENAI_API_KEY = carregar_chave_api("OPENAI_API_KEY")
print(f"  Prévia: {mascarar_chave(OPENAI_API_KEY)}")
print()

# Carregamento da chave do Google Gemini
# (sua chave pessoal gratuita do Google AI Studio)
# NOTA: Se você ainda não tem a chave do Google, não se preocupe!
#       Ela só é necessária na seção 8 (Módulo Gemini).
#       Você pode pular este passo por agora e voltar depois.
try:
    print("Carregando GOOGLE_API_KEY...")
    GOOGLE_API_KEY = carregar_chave_api("GOOGLE_API_KEY")
    print(f"  Prévia: {mascarar_chave(GOOGLE_API_KEY)}")
except (ValueError, EOFError):
    GOOGLE_API_KEY = None
    print("  ⚠️ Chave Google não configurada — tudo bem!")
    print("  Você pode continuar normalmente. A chave só é necessária na seção 8 (Gemini).")
    print("  Quando quiser usar o Gemini, configure a chave e rode esta célula novamente.")
print()

print("=" * 40)
if GOOGLE_API_KEY:
    print("Ambas as chaves estão prontas para uso!")
else:
    print("Chave OpenAI pronta! (Google Gemini: configurar depois, na seção 8)")

---

**Ambiente configurado!** Na próxima seção, vamos carregar e explorar o dataset de notícias.


# 3. Carregamento e Exploração dos Dados

## 3.1 Carregando o Dataset

Nosso dataset é um arquivo Excel (`Historias Para classificação.xlsx`) contendo **50 notícias econômicas brasileiras** já classificadas manualmente por analistas do IBRE/FGV. Essas classificações humanas servirão como nosso **ground truth** (verdade de referência) para avaliar o desempenho dos LLMs.

O arquivo contém as seguintes colunas:

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `date` | Data | Data de publicação da notícia |
| `jornal` | Texto | Nome do veículo de imprensa (ex: Folha de S.Paulo, Correio Braziliense) |
| `noticia` | Texto | Texto completo da notícia — este é o **input principal** para os LLMs |
| `incerteza` | Numérico (1/0) | A notícia transmite **incerteza econômica**? 1 = sim, 0 = não |
| `polaridade` | Numérico (-1/0/1) | Tom econômico da notícia: -1 = negativo, 0 = neutro, 1 = positivo |
| `internacional` | Texto ("sim"/NaN) | A notícia trata de um evento **internacional**? "sim" ou vazio (= nacional) |

> **Nota:** A coluna `polaridade` só é preenchida quando `incerteza = 1` (ou seja, quando a notícia de fato transmite incerteza econômica). Notícias sem incerteza terão `polaridade` como NaN.

In [6]:
# ============================================================
# 3.1 — Carregando o dataset de notícias
# ============================================================
# O arquivo pode estar no Google Drive (Colab) ou na pasta local.
# No Colab, se o arquivo não estiver disponível, ele é baixado
# automaticamente do GitHub.
# ============================================================

import os
import pandas as pd

# --- Caminhos possíveis para o arquivo ---
CAMINHO_DRIVE = "/content/drive/MyDrive/IBRE/Historias Para classificação.xlsx"
CAMINHO_LOCAL = "Historias Para classificação.xlsx"
CAMINHO_GITHUB = "https://raw.githubusercontent.com/OttoBoop/curso-pln-fgv-e-gfv/main/aula-01-classificacao/Historias%20Para%20classifica%C3%A7%C3%A3o.xlsx"

# Detecta qual caminho está disponível
if os.path.exists(CAMINHO_DRIVE):
    caminho_arquivo = CAMINHO_DRIVE
    print(f"Arquivo encontrado no Google Drive: {CAMINHO_DRIVE}")
elif os.path.exists(CAMINHO_LOCAL):
    caminho_arquivo = CAMINHO_LOCAL
    print(f"Arquivo encontrado localmente: {CAMINHO_LOCAL}")
elif AMBIENTE == "colab":
    # Download automático do GitHub quando no Colab
    import urllib.request
    print("Arquivo não encontrado localmente. Baixando do GitHub...")
    urllib.request.urlretrieve(CAMINHO_GITHUB, CAMINHO_LOCAL)
    caminho_arquivo = CAMINHO_LOCAL
    print(f"Download concluído: {CAMINHO_LOCAL}")
else:
    raise FileNotFoundError(
        "Arquivo 'Historias Para classificação.xlsx' não encontrado!\n"
        "  Certifique-se de que o arquivo está na mesma pasta deste notebook.\n"
        "  Dica: baixe a pasta completa da aula em:\n"
        "  https://github.com/OttoBoop/curso-pln-fgv-e-gfv"
    )

# Carrega o Excel em um DataFrame
df_noticias = pd.read_excel(caminho_arquivo)

# Visão rápida do dataset
print(f"\nDataset carregado com sucesso!")
print(f"  Dimensões: {df_noticias.shape[0]} linhas x {df_noticias.shape[1]} colunas")
print(f"  Colunas: {list(df_noticias.columns)}\n")

# Exibe as primeiras linhas
df_noticias.head()

Arquivo não encontrado localmente. Baixando do GitHub...
Download concluído: Historias Para classificação.xlsx

Dataset carregado com sucesso!
  Dimensões: 50 linhas x 6 colunas
  Colunas: ['date', 'jornal', 'noticia', 'incerteza', 'polaridade', 'internacional']



,date,jornal,noticia,incerteza,polaridade,internacional
0,2013-12-29,folha_impresso,\nGOVERNO PERDEU A BATALHA CONTRA O MERCADO ...,1,-1.0,NaN
1,2009-05-25,correio,MINISTROS DE ENERGIA DO G8 PEDEM MAIS INVESTIM...,1,-1.0,NaN
2,2021-10-04,folha_online,OPINIÃO - GIULIANA VALLONE: AMANHECER PÓS-PAND...,1,-1.0,NaN
3,2011-08-23,folha_online,"ELEONORA DE LUCENA\nDE SÃO PAULO\n""AS CRISES S...",1,-1.0,NaN
4,2014-01-29,valor_impresso,"ESTOU CONFIANTE EM RELAÇÃO AO BRASIL, DIZ CHEF...",1,1.0,NaN


In [ ]:
# ============================================================
# 3.2 — Exploração do dataset
# ============================================================
# Vamos entender a estrutura, tipos de dados, distribuição das
# classificações e verificar valores ausentes.
# ============================================================

print("=" * 60)
print("VISAO GERAL DO DATASET")
print("=" * 60)
print(f"\nFormato: {df_noticias.shape[0]} noticias x {df_noticias.shape[1]} colunas\n")

# --- Tipos de dados ---
print("-" * 60)
print("TIPOS DE DADOS POR COLUNA")
print("-" * 60)
print(df_noticias.dtypes.to_string())

# --- Valores ausentes (NaN) por coluna ---
print("\n" + "-" * 60)
print("VALORES AUSENTES (NaN) POR COLUNA")
print("-" * 60)
nulos_por_coluna = df_noticias.isnull().sum()
for coluna, qtd_nulos in nulos_por_coluna.items():
    marcador = " <--" if qtd_nulos > 0 else ""
    print(f"  {coluna:20s}: {qtd_nulos:3d} NaN(s){marcador}")

# --- Distribuição das colunas de classificação (ground truth) ---
colunas_classificacao = ["incerteza", "polaridade", "internacional"]

print("\n" + "=" * 60)
print("DISTRIBUICAO DAS CLASSIFICACOES (GROUND TRUTH)")
print("=" * 60)

for coluna in colunas_classificacao:
    print(f"\n--- {coluna} ---")
    contagem = df_noticias[coluna].value_counts(dropna=False).sort_index()
    for valor, qtd in contagem.items():
        rotulo = "NaN" if pd.isna(valor) else valor
        barra = "#" * int(qtd)
        print(f"  {str(rotulo):>10s}: {qtd:3d}  {barra}")

# --- Exemplos de notícias ---
print("\n" + "=" * 60)
print("EXEMPLOS DE NOTICIAS (texto truncado em 300 caracteres)")
print("=" * 60)

TAMANHO_TRECHO = 300

for i, linha in df_noticias.head(2).iterrows():
    texto = str(linha["noticia"])
    trecho = texto[:TAMANHO_TRECHO] + ("..." if len(texto) > TAMANHO_TRECHO else "")
    print(f"\nNoticia {i + 1} | Jornal: {linha['jornal']} | Data: {linha['date']}")
    print(f"  Incerteza: {linha['incerteza']} | "
          f"Polaridade: {linha['polaridade']} | "
          f"Internacional: {linha['internacional']}")
    print(f"  Texto: {trecho}")

### Observações sobre o dataset

**Tamanho:** O dataset contém apenas **50 notícias** — é um conjunto pequeno, ideal para experimentação manual e análise qualitativa, mas não para treinar modelos. Nosso objetivo aqui é **avaliar** a capacidade de LLMs pré-treinados, não treiná-los.

**Distribuição das classificações:**
- **Incerteza:** A maioria das notícias (34 de 50) foi classificada como transmitindo incerteza econômica (`incerteza = 1`). Isso reflete o viés de seleção — o IBRE monitora especificamente notícias sobre incerteza.
- **Polaridade:** Entre as notícias com incerteza, a grande maioria tem tom **negativo** (29). Poucas são neutras (2) ou positivas (3). As 16 notícias sem incerteza têm polaridade como NaN.
- **Internacional:** Apenas 9 notícias tratam de eventos internacionais. As demais (41) são nacionais (representadas como NaN no Excel).

**Valores ausentes (NaN):**
- `polaridade` tem 16 NaNs — são as notícias onde `incerteza = 0` (sem incerteza, não se avalia polaridade).
- `internacional` tem 41 NaNs — estes **não são dados faltantes**, mas sim a codificação de "nacional" (apenas "sim" foi marcado explicitamente).

> **Nota sobre NaN na coluna `internacional`:** No arquivo Excel, NaN na coluna `internacional` significa que a notícia é **nacional**. Apenas notícias internacionais receberam o valor "sim". Essa convenção será importante quando compararmos com as respostas dos LLMs.

## 3.3 As 4 Dimensões de Classificação

Cada notícia será classificada pelos LLMs em **4 dimensões** independentes. Abaixo detalhamos cada uma, com seus valores possíveis e o que significam economicamente:

| # | Dimensão | Pergunta que responde | Valores possíveis | Codificação |
|---|----------|----------------------|-------------------|-------------|
| 1 | **Incerteza** (`incerteza`) | A notícia transmite **incerteza econômica**? | Sim / Não | `1` = sim, `0` = não |
| 2 | **Polaridade** (`polaridade`) | Qual o **tom econômico** da notícia? | Negativo / Neutro / Positivo | `-1` = negativo, `0` = neutro, `1` = positivo |
| 3 | **Internacional** (`internacional`) | A notícia trata de evento **internacional**? | Sim / Não | `"sim"` = internacional, NaN = nacional |
| 4 | **Afeta o Brasil** (`inc_afeta_br`) | Se internacional, **afeta a economia brasileira**? | Sim / Não / N/A | `1` = sim, `0` = não, `None` = não se aplica |

> **Sobre a 4ª dimensão:** A coluna `inc_afeta_br` **não existe no arquivo Excel** — ela aparece apenas na saída da API quando enviamos notícias para classificação via LLM. Isso significa que não temos ground truth humano para essa dimensão, então ela não será avaliada quantitativamente, mas seus resultados ainda são interessantes para análise qualitativa.

### Relações entre as dimensões

As dimensões **não são totalmente independentes**. Existem relações lógicas:

- `polaridade` só faz sentido quando `incerteza = 1` (se não há incerteza, não se avalia o tom)
- `inc_afeta_br` só faz sentido quando `internacional = "sim"` (se a notícia é nacional, não cabe perguntar se "afeta o Brasil")

Essas dependências condicionais são incorporadas no prompt que enviamos ao LLM.

---

### Modificando as Dimensões

As dimensões acima são as que o IBRE utiliza atualmente, mas **você pode adaptá-las** ao seu caso de uso. O notebook foi projetado para ser flexível.

**Onde as dimensões são definidas?**

As dimensões de classificação são controladas em **dois lugares** (que veremos nas próximas seções):

1. **No prompt do LLM** (Seção 4) — onde descrevemos em linguagem natural o que o modelo deve classificar e quais valores são válidos. É aqui que o LLM "aprende" o que cada dimensão significa.

2. **Na função de parsing** (Seção 4, subseção 3) — onde extraímos e validamos as respostas do LLM, convertendo texto em dados estruturados.

**Exemplos de modificações possíveis:**

- **Adicionar uma dimensão:** ex: `setor` (agropecuária, indústria, serviços, financeiro)
- **Modificar valores:** ex: mudar `polaridade` para uma escala de 1 a 5 em vez de -1/0/1
- **Remover uma dimensão:** ex: remover `internacional` se todas as notícias forem nacionais
- **Criar dimensões temáticas:** ex: `tema` (inflação, câmbio, emprego, política fiscal, etc.)

> **Dica para voluntários:** Ao modificar dimensões, lembre-se de atualizar **tanto o prompt quanto a função de parsing**. Nas seções correspondentes, indicaremos com comentários `# >>> PERSONALIZAVEL <<<` os trechos que você pode alterar.

---

# 4. Classificação com GPT-3.5-Turbo (Exemplo Testado)

Este módulo reproduz a abordagem **original** utilizada na pesquisa do IBRE/FGV para classificar notícias econômicas brasileiras. Trata-se de um registro histórico e didático: o código aqui apresentado foi efetivamente utilizado para gerar os resultados da pesquisa.

---

### A Chat Completions API da OpenAI

A API da OpenAI para modelos de chat (como o `gpt-3.5-turbo`) funciona com um sistema de **mensagens em turnos**, onde cada mensagem possui um **papel** (*role*):

| Papel | Descrição | Exemplo |
|-------|-----------|---------|
| `system` | Define o comportamento geral do modelo. É a "personalidade" ou "instrução de fundo". O modelo tenta seguir essas diretrizes em todas as respostas. | *"Você é um economista brasileiro..."* |
| `user` | A mensagem do usuário — a pergunta ou tarefa que queremos que o modelo resolva. | *"Classifique esta notícia..."* |
| `assistant` | A resposta do modelo. Também pode ser fornecida manualmente para dar exemplos de respostas desejadas (*few-shot prompting*). | *"1. A notícia é nacional..."* |

### O que é um System Prompt?

O **system prompt** é a primeira mensagem da conversa e funciona como um "briefing" para o modelo. Ele:

- **Define o papel** que o modelo deve assumir (ex.: economista, tradutor, programador)
- **Estabelece restrições** de comportamento (ex.: responder apenas em português, ser conciso)
- **Contextualiza** o domínio do problema (ex.: análise de incerteza econômica)

Na pesquisa original, o system prompt escolhido foi:

> *"Você é um economista brasileiro especialista em analisar a persepção de incerteza dentro da economia"*

Esse prompt posiciona o modelo como um especialista no domínio, o que melhora a qualidade das respostas para tarefas de classificação econômica.

> **Nota sobre este módulo:** Este módulo usa `gpt-3.5-turbo` e a **Chat Completions API** — a interface clássica da OpenAI. Nas seções seguintes, apresentaremos abordagens mais modernas: a **Responses API** da OpenAI (com saída estruturada nativa) e o **Gemini** do Google, que eliminam várias das limitações que veremos aqui.

## 4.1 Função de Chamada à API

Vamos criar uma função reutilizável que encapsula a chamada à API da OpenAI. Essa função:

- **Recebe** um prompt (texto) e envia ao modelo `gpt-3.5-turbo`
- **Inclui** o system prompt que define o papel de economista brasileiro
- **Retorna** a resposta do modelo como texto puro (string)
- **Trata erros** comuns de forma educativa, para facilitar a depuração

### Parâmetros importantes

- **`temperatura`** (padrão: `0`): Controla a aleatoriedade da resposta. Com `temperatura=0`, o modelo tende a dar sempre a mesma resposta para o mesmo input — ideal para classificação, onde queremos consistência e reprodutibilidade. Valores mais altos (ex.: 0.7) geram respostas mais criativas e variadas.

- **`model`**: Usamos `gpt-3.5-turbo`, o modelo utilizado na pesquisa original. Era o melhor custo-benefício disponível na época.

In [ ]:
import time
from openai import OpenAI

# ============================================================
# Função de chamada à API — Chat Completions (gpt-3.5-turbo)
# ============================================================

# Criamos o cliente da OpenAI usando a chave já carregada na seção de Setup.
# A variável OPENAI_API_KEY foi definida anteriormente neste notebook.
cliente_openai = OpenAI(api_key=OPENAI_API_KEY)

# System prompt original da pesquisa IBRE/FGV.
# Nota: mantemos o texto exatamente como foi usado, incluindo a grafia "persepção".
SYSTEM_PROMPT = (
    "Você é um economista brasileiro especialista em analisar "
    "a persepção de incerteza dentro da economia"
)


def call_api(prompt: str, temperatura: float = 0) -> str:
    """
    Envia um prompt ao modelo gpt-3.5-turbo e retorna a resposta como texto.

    Parâmetros
    ----------
    prompt : str
        O texto da mensagem do usuário (a tarefa de classificação).
    temperatura : float, opcional
        Controla a aleatoriedade da resposta (0 = determinístico). Padrão: 0.

    Retorna
    -------
    str
        O texto da resposta do modelo, sem espaços extras nas pontas.
        Em caso de erro, retorna uma string descrevendo o problema.
    """
    try:
        resposta = cliente_openai.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            temperature=temperatura
        )
        texto_resposta = resposta.choices[0].message.content.strip()
        return texto_resposta

    except Exception as erro:
        tipo_erro = type(erro).__name__

        if "RateLimitError" in tipo_erro:
            print("ERRO DE LIMITE DE REQUISICOES (Rate Limit):")
            print("  A API tem limites de quantas requisições você pode fazer por minuto.")
            print("  Solução: aguarde alguns segundos e tente novamente.")
            print(f"  Detalhe técnico: {erro}")
            return "ERRO: Rate limit atingido. Aguarde e tente novamente."

        elif "AuthenticationError" in tipo_erro:
            print("ERRO DE AUTENTICACAO:")
            print("  Sua chave de API (OPENAI_API_KEY) está inválida ou expirada.")
            print("  Solução: verifique a chave na seção de Setup deste notebook.")
            print(f"  Detalhe técnico: {erro}")
            return "ERRO: Falha na autenticação. Verifique sua API key."

        elif "Timeout" in tipo_erro or "TimeoutError" in tipo_erro:
            print("ERRO DE TIMEOUT:")
            print("  A requisição demorou mais do que o esperado para obter resposta.")
            print("  Solução: aguarde alguns segundos e tente novamente.")
            print(f"  Detalhe técnico: {erro}")
            return "ERRO: Timeout na requisição. Tente novamente."

        else:
            print(f"ERRO INESPERADO ({tipo_erro}):")
            print(f"  {erro}")
            print("  Se o erro persistir, verifique sua conexão com a internet")
            print("  e o status da API em: https://status.openai.com/")
            return f"ERRO: {tipo_erro} - {erro}"


# --- Teste rápido da função ---
# Descomente as linhas abaixo para verificar se a API está funcionando:
# resposta_teste = call_api("Responda apenas: 'API funcionando!'")
# print(f"Resposta do teste: {resposta_teste}")

print("Funcao 'call_api' definida com sucesso.")
print(f"  Modelo: gpt-3.5-turbo")
print(f"  System prompt: '{SYSTEM_PROMPT[:60]}...'")

## 4.2 O Prompt de Classificação

O prompt é a parte **mais importante** de todo este processo. É nele que dizemos ao modelo *exatamente* o que queremos. Um bom prompt de classificação precisa:

1. **Definir a tarefa claramente** — o modelo precisa saber que deve *classificar*, não resumir ou comentar
2. **Especificar as dimensões** — quais perguntas queremos que ele responda
3. **Exigir um formato exato** — como o modelo retorna texto livre, precisamos que ele siga um padrão que possamos extrair programaticamente
4. **Fornecer o conteúdo** — a notícia a ser classificada

### Decisões de engenharia de prompt

- **Formato rígido**: Pedimos que o modelo responda com frases *exatas*, pois depois usaremos regex para extrair as respostas. Qualquer desvio do formato pode causar falhas na extração.
- **Aviso explícito**: Incluímos um aviso de que "suas respostas serão lidas por um programa de Python" — isso incentiva o modelo a ser mais rigoroso com o formato.
- **Pergunta condicional**: A pergunta 3 só se aplica quando a notícia é internacional E demonstra incerteza, por isso incluímos a opção "N/A".

> **Para voluntários:** Este prompt é o componente que mais influencia a qualidade dos resultados. Na seção "Agora É Sua Vez", você terá espaço para modificar as instruções, adicionar exemplos (*few-shot*) ou reformular as perguntas para ver como o modelo responde de forma diferente.

In [ ]:
# ============================================================
# Prompt de classificação — template com as 4 dimensões
# ============================================================

def construir_prompt(noticia: str) -> str:
    """
    Constrói o prompt completo de classificação para uma notícia.

    O prompt pede ao modelo que avalie a notícia em 4 dimensões e
    responda seguindo um formato rígido para facilitar a extração
    automática via regex.

    Parâmetros
    ----------
    noticia : str
        O texto da notícia a ser classificada.

    Retorna
    -------
    str
        O prompt completo, pronto para ser enviado à API.
    """

    # >>> PERSONALIZAVEL <<< — Modifique as perguntas e formatos abaixo
    prompt = (
        # --- Seção 1: Definição da tarefa e das 4 perguntas ---
        "Leia e avalie a seguinte notícia sob as seguintes métricas:\n"
        "1. A notícia é internacional (fala sobre países além do Brasil) ou nacional?\n"
        "2. Ela demonstra incerteza sobre a economia do país ou países aos quais ela referencia?\n"
        "3. Se a notícia fala de um país além do Brasil e demonstra incerteza sobre a economia "
        "desse país, ela diz que essa incerteza afeta a economia brasileira negativamente?\n"
        "4. Essa notícia tem tom positivo, negativo, ou neutro?\n"
        "\n"

        # --- Seção 2: A notícia em si ---
        f"{noticia}\n"
        "\n"

        # --- Seção 3: Instruções de formato rígido ---
        "É de extrema importância que você responda as quatro perguntas seguindo "
        "EXATAMENTE o padrão que lhe detalharei, pois suas respostas serão lidas por "
        "um programa de Python, que precisa ler trechos exatos para funcionar corretamente.\n"
        "\n"

        # >>> PERSONALIZAVEL <<< — Pergunta 1: Escopo geográfico
        "Primeiro, se a notícia é nacional escreva EXATAMENTE: '1. A notícia é nacional'.\n"
        "Se a notícia é internacional, escreva EXATAMENTE: '1. A notícia é internacional'.\n"
        "\n"

        # >>> PERSONALIZAVEL <<< — Pergunta 2: Incerteza econômica
        "Sobre a incerteza sobre a economia do país, se a notícia demonstra incerteza, "
        "escreva EXATAMENTE: "
        "'2. Sim, a notícia demonstra incerteza sobre a economia dos países analisados'.\n"
        "Se a notícia não demonstra incerteza, escreva EXATAMENTE: "
        "'2. Não, a notícia não demonstra incerteza sobre a economia dos países analisados'.\n"
        "\n"

        # >>> PERSONALIZAVEL <<< — Pergunta 3: Impacto no Brasil
        "Caso a notícia envolva países além do Brasil e também envolva incerteza, "
        "e essa incerteza afeta negativamente a economia brasileira, "
        "escreva EXATAMENTE: '3. A incerteza envolve negativamente a economia brasileira'.\n"
        "Se a incerteza não afeta a economia brasileira, escreva EXATAMENTE: "
        "'3. A incerteza não envolve negativamente a economia brasileira'.\n"
        "Se a questão sobre a incerteza econômica não se aplica, escreva EXATAMENTE: "
        "'3. A questão 3 não se aplica'.\n"
        "\n"

        # >>> PERSONALIZAVEL <<< — Pergunta 4: Tom/polaridade
        "Por fim, sobre o tom da notícia, se possui tom positivo, escreva EXATAMENTE: "
        "'4. A notícia possui tom positivo'.\n"
        "Se possui tom negativo, escreva EXATAMENTE: '4. A notícia possui tom negativo'.\n"
        "Se possui tom neutro, escreva EXATAMENTE: '4. A notícia possui tom neutro'."
    )

    return prompt


# --- Visualização do prompt para a primeira notícia (demonstração) ---
exemplo_noticia = df_noticias.iloc[0]["noticia"]
exemplo_prompt = construir_prompt(exemplo_noticia)

print("=" * 70)
print("EXEMPLO DE PROMPT CONSTRUIDO (primeiros 500 caracteres)")
print("=" * 70)
print(exemplo_prompt[:500])
print("...")
print(f"\n[Tamanho total do prompt: {len(exemplo_prompt)} caracteres]")

## 4.3 Extraindo Respostas com Regex

### O problema: texto livre vs. dados estruturados

O modelo `gpt-3.5-turbo` retorna **texto livre** — uma string de texto como qualquer resposta de chat. Porém, para análise de dados, precisamos de **valores estruturados** (números, categorias) que possamos colocar em um DataFrame.

Por exemplo, o modelo pode retornar:

```
1. A notícia é internacional
2. Sim, a notícia demonstra incerteza sobre a economia dos países analisados
3. A incerteza envolve negativamente a economia brasileira
4. A notícia possui tom negativo
```

E precisamos extrair disso: `{"internacional": 1, "incerteza": 1, "inc_afeta_br": 1, "polaridade": -1}`

### O que é Regex?

**Regex** (Regular Expressions / Expressões Regulares) é uma linguagem de busca de padrões em texto. Em Python, usamos o módulo `re` para isso. Alguns exemplos:

| Padrão | Significado | Exemplo de match |
|--------|-------------|------------------|
| `r"A notícia é (nacional\|internacional)"` | Busca a frase e captura qual das duas opções aparece | "A notícia é **internacional**" |
| `r"tom (positivo\|negativo\|neutro)"` | Busca "tom" seguido de uma das 3 opções | "tom **negativo**" |
| `re.IGNORECASE` | Flag que ignora maiúsculas/minúsculas | "NACIONAL" casa com "nacional" |

O método `re.search(padrão, texto)` retorna um objeto *match* se encontrar o padrão, ou `None` se não encontrar. Usamos `.group(1)` para acessar o conteúdo capturado entre parênteses.

> **Nota importante:** Nos módulos modernos (Responses API e Gemini), veremos como **structured outputs** eliminam completamente a necessidade de regex. O modelo retorna JSON validado diretamente, sem necessidade de parsing manual. Esta é uma das principais motivações para migrar para abordagens mais modernas.

In [ ]:
import re

# ============================================================
# Extração de classificações com Regex
# ============================================================

def extrair_classificacoes(resposta_api: str) -> dict:
    """
    Extrai as 4 dimensões de classificação da resposta textual do modelo.

    Usa expressões regulares (regex) para buscar padrões específicos
    no texto retornado pela API e converte em valores numéricos.

    Parâmetros
    ----------
    resposta_api : str
        O texto completo retornado pelo modelo gpt-3.5-turbo.

    Retorna
    -------
    dict
        Dicionário com as chaves:
        - 'internacional': 1 (internacional), 0 (nacional), ou "erro"
        - 'incerteza': 1 (sim), 0 (não), ou "erro"
        - 'inc_afeta_br': 1 (sim), 0 (não), None (N/A), ou "erro"
        - 'polaridade': 1 (positivo), 0 (neutro), -1 (negativo), ou "erro"
    """

    # ----------------------------------------------------------
    # Dimensão 1: Escopo geográfico (nacional ou internacional)
    # ----------------------------------------------------------
    # >>> PERSONALIZAVEL <<< — Adapte o regex se mudar o formato da pergunta 1
    match_internacional = re.search(
        r"1\. A notícia é (nacional|internacional)",
        resposta_api,
        re.IGNORECASE
    )

    if match_internacional:
        valor_capturado = match_internacional.group(1).lower()
        internacional = 1 if valor_capturado == "internacional" else 0
    else:
        internacional = "erro"

    # ----------------------------------------------------------
    # Dimensão 2: Demonstra incerteza econômica (sim ou não)
    # ----------------------------------------------------------
    # >>> PERSONALIZAVEL <<< — Adapte o regex se mudar o formato da pergunta 2
    match_incerteza_sim = re.search(
        r"2\. Sim, a notícia demonstra incerteza",
        resposta_api,
        re.IGNORECASE
    )
    match_incerteza_nao = re.search(
        r"2\. Não, a notícia não demonstra incerteza",
        resposta_api,
        re.IGNORECASE
    )

    if match_incerteza_sim:
        incerteza = 1
    elif match_incerteza_nao:
        incerteza = 0
    else:
        incerteza = "erro"

    # ----------------------------------------------------------
    # Dimensão 3: Incerteza internacional afeta o Brasil
    # ----------------------------------------------------------
    # >>> PERSONALIZAVEL <<< — Adapte o regex se mudar o formato da pergunta 3
    # Importante: checamos "não" ANTES de "sim" porque a string "sim"
    # é substring de "não envolve negativamente" — ordem importa!
    match_afeta_nao = re.search(
        r"3\. A incerteza não envolve negativamente a economia brasileira",
        resposta_api,
        re.IGNORECASE
    )
    match_afeta_sim = re.search(
        r"3\. A incerteza envolve negativamente a economia brasileira",
        resposta_api,
        re.IGNORECASE
    )
    match_afeta_na = re.search(
        r"3\. A questão 3 não se aplica",
        resposta_api,
        re.IGNORECASE
    )

    if match_afeta_nao:
        inc_afeta_br = 0
    elif match_afeta_sim:
        inc_afeta_br = 1
    elif match_afeta_na:
        inc_afeta_br = None  # None = "não se aplica"
    else:
        inc_afeta_br = "erro"

    # ----------------------------------------------------------
    # Dimensão 4: Tom/Polaridade (positivo, negativo, ou neutro)
    # ----------------------------------------------------------
    # >>> PERSONALIZAVEL <<< — Adapte o regex se mudar o formato da pergunta 4
    match_polaridade = re.search(
        r"4\. A notícia possui tom (positivo|negativo|neutro)",
        resposta_api,
        re.IGNORECASE
    )

    if match_polaridade:
        tom_capturado = match_polaridade.group(1).lower()
        polaridade = {"positivo": 1, "neutro": 0, "negativo": -1}[tom_capturado]
    else:
        polaridade = "erro"

    return {
        "internacional": internacional,
        "incerteza": incerteza,
        "inc_afeta_br": inc_afeta_br,
        "polaridade": polaridade
    }


# --- Teste da função com uma resposta simulada ---
resposta_exemplo = """
1. A notícia é internacional
2. Sim, a notícia demonstra incerteza sobre a economia dos países analisados
3. A incerteza envolve negativamente a economia brasileira
4. A notícia possui tom negativo
"""

resultado_teste = extrair_classificacoes(resposta_exemplo)
esperado = {"internacional": 1, "incerteza": 1, "inc_afeta_br": 1, "polaridade": -1}

print("Teste de extracao com resposta simulada:")
print(f"  Resultado: {resultado_teste}")
print(f"  Esperado:  {esperado}")
print(f"  Teste OK:  {resultado_teste == esperado}")

## 4.4 Executando a Classificação

Agora vamos juntar todas as peças: para cada notícia, construímos o prompt, enviamos à API, e extraímos as classificações da resposta.

### Duas opções de execução

| Opção | Notícias | Tempo estimado | Custo | Quando usar |
|-------|----------|----------------|-------|-------------|
| **Teste rápido** | 5 primeiras | ~30 segundos | Centavos | Para validar que o pipeline funciona |
| **Completo** | Todas (50) | ~3-5 minutos | Baixo (~US$ 0.05) | Quando quiser avaliar as métricas reais |

> **Atenção sobre custos e tempo:**
> - Cada chamada à API consome tokens (e portanto créditos)
> - O `gpt-3.5-turbo` é relativamente barato, mas o custo acumula com muitas notícias
> - Incluímos `time.sleep(1)` entre chamadas para respeitar os limites de requisições por minuto (*rate limits*)
> - Se a execução for interrompida, os resultados parciais ficam preservados na lista

In [ ]:
# ============================================================
# Teste Rapido (5 noticias)
# ============================================================
# Executamos apenas as 5 primeiras notícias para validar que
# todo o pipeline funciona antes de rodar o dataset completo.
# ============================================================

resultados_teste = []

print("Iniciando teste rapido com 5 noticias...")
print("=" * 50)

for indice, linha in df_noticias.head(5).iterrows():
    noticia = linha["noticia"]

    # Passo 1: Construir o prompt
    prompt = construir_prompt(noticia)

    # Passo 2: Enviar à API
    print(f"\nNoticia {indice + 1}/5 - Enviando a API...", end=" ")
    resposta = call_api(prompt)

    # Verificar se a API retornou erro
    if resposta.startswith("ERRO:"):
        print(f"FALHA na API!")
        print(f"  Resposta: {resposta}")

    # Passo 3: Extrair classificações da resposta
    classificacoes = extrair_classificacoes(resposta)

    # Verificar se houve erros na extração
    erros = [chave for chave, valor in classificacoes.items() if valor == "erro"]
    if erros and not resposta.startswith("ERRO:"):
        print(f"Extracao parcial - campos com erro: {erros}")
    elif not resposta.startswith("ERRO:"):
        print("OK")

    # Armazenar resultado
    classificacoes["indice_original"] = indice
    classificacoes["resposta_api_bruta"] = resposta
    resultados_teste.append(classificacoes)

    # Pausa entre chamadas para respeitar rate limits
    time.sleep(1)

print("\n" + "=" * 50)
print(f"Teste concluido! {len(resultados_teste)} noticias classificadas.")

# --- Exibição dos resultados ---
df_teste = pd.DataFrame(resultados_teste)
print("\nResultados do teste rapido:")
colunas_exibir = ["indice_original", "internacional", "incerteza", "inc_afeta_br", "polaridade"]
print(df_teste[colunas_exibir].to_string(index=False))

# Contagem de erros
total_campos = len(resultados_teste) * 4
total_erros = sum(
    1 for r in resultados_teste
    for chave in ["internacional", "incerteza", "inc_afeta_br", "polaridade"]
    if r[chave] == "erro"
)
print(f"\nTotal de erros de extracao: {total_erros} de {total_campos} campos")

In [ ]:
# ============================================================
# Classificacao Completa (todas as noticias)
# ============================================================
#
# ATENCAO: Esta célula processa TODAS as notícias do dataset.
# - Pode levar vários minutos dependendo do tamanho
# - Cada notícia gera uma chamada à API (custo de tokens)
# - A pausa de 1 segundo entre chamadas evita rate limits
#
# Mude EXECUTAR_CLASSIFICACAO_COMPLETA para True quando estiver
# pronto para rodar o pipeline completo.
# ============================================================

EXECUTAR_CLASSIFICACAO_COMPLETA = False  # <-- Mude para True para executar

if EXECUTAR_CLASSIFICACAO_COMPLETA:

    resultados_classificacao = []
    total_noticias = len(df_noticias)

    print(f"Iniciando classificacao de {total_noticias} noticias...")
    print(f"Tempo estimado: ~{total_noticias * 2} segundos ({total_noticias * 2 / 60:.1f} minutos)")
    print("=" * 60)

    for indice, linha in df_noticias.iterrows():
        noticia = linha["noticia"]

        # Passo 1: Construir o prompt
        prompt = construir_prompt(noticia)

        # Passo 2: Enviar à API
        progresso = f"[{indice + 1}/{total_noticias}]"
        print(f"{progresso} Classificando...", end=" ")
        resposta = call_api(prompt)

        # Verificar erro da API
        if resposta.startswith("ERRO:"):
            print(f"FALHA: {resposta[:60]}")

        # Passo 3: Extrair classificações
        classificacoes = extrair_classificacoes(resposta)

        # Verificar erros de extração
        erros = [k for k, v in classificacoes.items() if v == "erro"]
        if erros and not resposta.startswith("ERRO:"):
            print(f"Parcial - erros: {erros}")
        elif not resposta.startswith("ERRO:"):
            print("OK")

        # Armazenar
        classificacoes["indice_original"] = indice
        classificacoes["resposta_api_bruta"] = resposta
        resultados_classificacao.append(classificacoes)

        # Pausa para respeitar rate limits
        time.sleep(1)

        # Progresso a cada 10 notícias
        if (indice + 1) % 10 == 0:
            print(f"--- Progresso: {indice + 1}/{total_noticias} "
                  f"({(indice + 1) / total_noticias * 100:.0f}%) ---")

    print("\n" + "=" * 60)
    print(f"Classificacao concluida! {len(resultados_classificacao)} noticias processadas.")

    # --- Criar DataFrame com os resultados ---
    df_resultados = pd.DataFrame(resultados_classificacao)
    df_resultados = df_resultados.set_index("indice_original")

    # Merge com o DataFrame original usando sufixo para evitar conflito
    df_classificado = df_noticias.join(
        df_resultados[["internacional", "incerteza", "inc_afeta_br", "polaridade"]],
        rsuffix="_modelo"
    )

    print("\nPrimeiras linhas do DataFrame combinado:")
    display(df_classificado.head())

    # --- Resumo de qualidade ---
    total_campos = len(resultados_classificacao) * 4
    total_erros = sum(
        1 for r in resultados_classificacao
        for k in ["internacional", "incerteza", "inc_afeta_br", "polaridade"]
        if r[k] == "erro"
    )
    print(f"\nResumo de qualidade:")
    print(f"  Total de campos classificados: {total_campos}")
    print(f"  Campos com erro de extracao:   {total_erros} ({total_erros/total_campos*100:.1f}%)")
    print(f"  Taxa de sucesso:               {(total_campos-total_erros)/total_campos*100:.1f}%")

else:
    print("Classificacao completa desativada.")
    print("  Para executar, altere EXECUTAR_CLASSIFICACAO_COMPLETA para True acima.")
    print("  Certifique-se de que o teste rapido (celula anterior) funcionou corretamente.")

---

## Nota Histórica: Por Que Regex?

Neste módulo, usamos **expressões regulares (regex)** para extrair dados estruturados da resposta do modelo. Isso levanta uma pergunta natural: *por que não pedir ao modelo que retorne JSON diretamente?*

### As limitações do gpt-3.5-turbo

Na época em que esta pesquisa foi conduzida, o `gpt-3.5-turbo` (via Chat Completions API) **não oferecia suporte nativo a saídas estruturadas**. O modelo sempre retornava texto livre, e mesmo pedindo explicitamente por JSON, o resultado podia:

- Incluir texto extra antes ou depois do JSON
- Conter erros de sintaxe no JSON (vírgulas faltando, aspas incorretas)
- Mudar o formato entre uma resposta e outra
- Ignorar parcialmente as instruções de formato

Por isso, a estratégia adotada foi pedir frases exatas e usar regex para extraí-las — uma abordagem que funciona, porém **frágil**: se o modelo mudar uma única palavra (ex.: "A notícia **tem** tom negativo" em vez de "A notícia **possui** tom negativo"), o regex falha e retorna `"erro"`.

### O que mudou desde então

As APIs modernas resolvem esse problema de forma elegante:

| Abordagem | API | Vantagem |
|-----------|-----|----------|
| **Structured Outputs** (OpenAI) | Responses API | O modelo é *obrigado* a retornar JSON válido seguindo um schema Pydantic |
| **JSON Mode** (Google) | Gemini API | Saída JSON nativa com validação de schema |

Nos próximos módulos, veremos como essas abordagens eliminam completamente a necessidade de regex, tornando o pipeline mais robusto, mais simples e mais fácil de manter.

---

**Próximas seções:** Vamos avaliar os resultados deste módulo (métricas) e depois reimplementar a classificação usando abordagens modernas.

# 5. Avaliação de Resultados

Agora que temos as classificações do modelo, precisamos responder à pergunta mais importante: **quão bom ele é?**

Para isso, vamos comparar as respostas do GPT-3.5-turbo com o **gabarito humano** — as classificações feitas manualmente que já existem no nosso DataFrame `df_noticias`.

Vamos avaliar três dimensões:
- **`incerteza`** — o artigo menciona incerteza econômica? (1 = sim, 0 = não)
- **`polaridade`** — o tom da incerteza é negativo, neutro ou positivo? (-1, 0, 1)
- **`internacional`** — a notícia trata de tema internacional? (1 = sim, 0 = não)

> **Nota:** A dimensão `inc_afeta_br` não possui gabarito humano no dataset original, então não será avaliada aqui.

A função abaixo é **reutilizável** — vamos chamá-la novamente mais adiante para comparar diferentes modelos e prompts.

In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score
import numpy as np

def calcular_metricas(df_gabarito, resultados, colunas_avaliar=None):
    """
    Compara as predições do modelo com o gabarito humano e calcula métricas.

    Parâmetros
    ----------
    df_gabarito : pd.DataFrame
        DataFrame original com as classificações humanas (ground truth).
    resultados : list[dict]
        Lista de dicionários retornada pela função de classificação.
        Cada dict deve ter 'indice_original' e as colunas de classificação.
    colunas_avaliar : list[str], opcional
        Quais dimensões avaliar. Padrão: ['incerteza', 'polaridade', 'internacional'].

    Retorna
    -------
    dict
        Dicionário com as métricas de cada dimensão, útil para comparações futuras.
    """
    if colunas_avaliar is None:
        colunas_avaliar = ['incerteza', 'polaridade', 'internacional']

    # --- 1. Montar DataFrame de predições alinhado ao gabarito ---
    df_predicoes = pd.DataFrame(resultados)

    if 'indice_original' not in df_predicoes.columns:
        print("Os resultados nao contem 'indice_original'. Nao e possivel alinhar com o gabarito.")
        return {}

    df_predicoes = df_predicoes.set_index('indice_original')

    # --- 2. Preparar o gabarito com conversões de encoding ---
    df_gab = df_gabarito.copy()

    # Converter 'internacional': "sim" → 1, NaN → 0
    if 'internacional' in df_gab.columns:
        df_gab['internacional'] = df_gab['internacional'].apply(
            lambda x: 1 if str(x).strip().lower() == 'sim' else 0
        )

    # --- 3. Calcular métricas para cada dimensão ---
    resumo = {}
    total_artigos = len(df_predicoes)
    print("=" * 60)
    print(f"  AVALIACAO DE METRICAS - {total_artigos} artigos classificados")
    print("=" * 60)

    for coluna in colunas_avaliar:
        if coluna not in df_gab.columns:
            print(f"\nColuna '{coluna}' nao encontrada no gabarito. Pulando...")
            continue
        if coluna not in df_predicoes.columns:
            print(f"\nColuna '{coluna}' nao encontrada nos resultados. Pulando...")
            continue

        # Alinhar pelos índices em comum
        indices_comuns = df_predicoes.index.intersection(df_gab.index)
        gabarito = df_gab.loc[indices_comuns, coluna]
        predicao = df_predicoes.loc[indices_comuns, coluna]

        # Filtrar: remover onde gabarito é NaN
        filtro_gab_valido = gabarito.notna()

        # Filtrar: remover onde predição é "erro" ou NaN
        filtro_pred_valido = predicao.apply(
            lambda x: x != "erro" and pd.notna(x)
        )

        filtro_final = filtro_gab_valido & filtro_pred_valido

        n_excluidos_gab = (~filtro_gab_valido).sum()
        n_excluidos_pred = (filtro_gab_valido & ~filtro_pred_valido).sum()
        n_validos = filtro_final.sum()

        print(f"\n{'-' * 60}")
        print(f"  Dimensao: {coluna.upper()}")
        print(f"{'-' * 60}")
        print(f"  Artigos com gabarito disponivel : {filtro_gab_valido.sum()}")
        print(f"  Excluidos (erro na extracao)    : {n_excluidos_pred}")
        print(f"  Artigos avaliados               : {n_validos}")

        if n_validos == 0:
            print("  Nenhum artigo valido para comparacao nesta dimensao.")
            resumo[coluna] = {
                'acuracia': None, 'precisao': None,
                'n_validos': 0, 'n_excluidos_erro': int(n_excluidos_pred),
                'n_excluidos_nan': int(n_excluidos_gab)
            }
            continue

        verdadeiros = gabarito[filtro_final].astype(float).values
        previstos = predicao[filtro_final].astype(float).values

        # Calcular métricas
        acuracia = accuracy_score(verdadeiros, previstos)
        precisao = precision_score(
            verdadeiros, previstos, average='weighted', zero_division=0
        )
        matriz = confusion_matrix(verdadeiros, previstos)

        # Exibir resultados
        print(f"\n  Acuracia : {acuracia:.1%}")
        print(f"  Precisao : {precisao:.1%} (media ponderada)")

        # Exibir matriz de confusão com rótulos
        rotulos = sorted(set(verdadeiros) | set(previstos))
        rotulos_str = [str(int(r)) for r in rotulos]
        print(f"\n  Matriz de Confusao (linhas=real, colunas=predicao):")
        cabecalho = "         " + "  ".join(f"pred={r:>4s}" for r in rotulos_str)
        print(f"  {cabecalho}")
        for i, rotulo in enumerate(rotulos_str):
            linha = f"  real={rotulo:>4s}  " + "  ".join(
                f"{matriz[i][j]:>7d}" for j in range(len(rotulos_str))
            )
            print(linha)

        resumo[coluna] = {
            'acuracia': acuracia,
            'precisao': precisao,
            'matriz_confusao': matriz,
            'rotulos': rotulos,
            'n_validos': int(n_validos),
            'n_excluidos_erro': int(n_excluidos_pred),
            'n_excluidos_nan': int(n_excluidos_gab)
        }

    print(f"\n{'=' * 60}")
    print("  Avaliacao concluida.")
    print(f"{'=' * 60}")

    return resumo

print("Funcao calcular_metricas() definida com sucesso.")

## O que cada métrica significa

Antes de rodar a avaliação, vale entender o que estamos medindo.

### Matriz de Confusão

A matriz de confusão cruza **o que o modelo disse** (predição) com **o que o gabarito humano diz** (real). Para um problema binário (sim/não), ela tem esta forma:

|  | **Pred = 0** | **Pred = 1** |
|---|:---:|:---:|
| **Real = 0** | Verdadeiro Negativo (VN) | Falso Positivo (FP) |
| **Real = 1** | Falso Negativo (FN) | Verdadeiro Positivo (VP) |

- **Verdadeiro Positivo (VP):** o modelo disse "sim" e o gabarito concorda.
- **Verdadeiro Negativo (VN):** o modelo disse "não" e o gabarito concorda.
- **Falso Positivo (FP):** o modelo disse "sim", mas o gabarito diz "não" (alarme falso).
- **Falso Negativo (FN):** o modelo disse "não", mas o gabarito diz "sim" (caso perdido).

### Acurácia

A acurácia é simplesmente: **acertos totais / total de artigos**.

**Exemplo concreto:** se temos 10 artigos e o modelo acertou 7, a acurácia é **70%**.

É a métrica mais intuitiva, mas pode enganar quando as classes são desbalanceadas. Se 90% dos artigos são "não incerteza", um modelo que sempre responde "não" teria 90% de acurácia — sem entender nada.

### Precisão (Ponderada)

A precisão responde: **das vezes que o modelo disse "sim", quantas vezes ele estava certo?**

Usamos a versão **ponderada** (`weighted`), que calcula a precisão de cada classe e faz a média ponderada pelo número de amostras em cada classe. Isso evita distorções quando uma classe tem muito mais exemplos que outra.

> **Dica:** Na dimensão `polaridade`, a matriz terá 3x3 em vez de 2x2, pois existem três valores possíveis (-1, 0, 1). A lógica é a mesma — cada valor vira uma "classe".

In [ ]:
# ============================================================
# Executando a avaliação de métricas
# ============================================================

# Detectar qual conjunto de resultados usar
if 'resultados_classificacao' in dir() and resultados_classificacao:
    resultados_para_avaliar = resultados_classificacao
    nome_conjunto = "classificacao completa"
elif 'resultados_teste' in dir() and resultados_teste:
    resultados_para_avaliar = resultados_teste
    nome_conjunto = "teste rapido (5 artigos)"
else:
    raise ValueError(
        "Nenhum resultado encontrado. Execute primeiro o Modulo 1 "
        "(teste rapido ou classificacao completa) antes de rodar esta celula."
    )

print(f"Usando resultados da: {nome_conjunto}")
print(f"Total de artigos nos resultados: {len(resultados_para_avaliar)}\n")

# Resumo de qualidade das extrações
colunas_modelo = ['incerteza', 'polaridade', 'internacional', 'inc_afeta_br']
print("Resumo de qualidade das extracoes:")
print("-" * 45)
for col in colunas_modelo:
    n_erro = sum(1 for r in resultados_para_avaliar if r.get(col) == "erro")
    n_none = sum(1 for r in resultados_para_avaliar if r.get(col) is None)
    n_ok = len(resultados_para_avaliar) - n_erro - n_none
    print(f"  {col:<16s}  OK: {n_ok:>3d}  |  erro: {n_erro:>3d}  |  None: {n_none:>3d}")
print("-" * 45)
print()

# Calcular métricas (sem inc_afeta_br, que não tem gabarito)
resumo_modulo1 = calcular_metricas(
    df_gabarito=df_noticias,
    resultados=resultados_para_avaliar,
    colunas_avaliar=['incerteza', 'polaridade', 'internacional']
)

### Entendendo os valores excluídos

Dois tipos de artigos são **excluídos** do cálculo das métricas:

- **`"erro"` na predição:** significa que o regex não conseguiu extrair o valor daquela dimensão da resposta da API. Isso pode acontecer quando o modelo retorna a resposta num formato inesperado. Quanto mais erros, mais vale revisar o prompt ou o regex de extração.

- **`NaN` no gabarito:** significa que o avaliador humano não classificou aquela dimensão para aquele artigo. Por exemplo, a coluna `polaridade` só é preenchida para artigos que mencionam incerteza — se `incerteza = 0`, a polaridade fica vazia. Esses artigos são excluídos da comparação, pois não temos contra o que comparar.

A dimensão **`inc_afeta_br`** (se a incerteza internacional afeta o Brasil) não possui nenhum gabarito humano no dataset, então ela aparece apenas nos resultados do modelo, sem avaliação de métricas.

> **Guarde a variável `resumo_modulo1`** — vamos usá-la mais adiante para comparar com outros modelos e prompts.

---

# 6. Agora É Sua Vez!

Até aqui, você viu como o pipeline funciona de ponta a ponta: construir um prompt, enviar à API, extrair as respostas com regex e medir a qualidade com métricas. O resultado do Módulo 1 está armazenado em `resumo_modulo1`.

Agora é a sua vez de **melhorar o prompt** e tentar superar o baseline. Abaixo estão 5 dicas práticas de prompt engineering para você experimentar.

---

### Dica 1: Seja mais específico — use exemplos no prompt (few-shot)

Inclua 1-2 exemplos de notícias já classificadas dentro do próprio prompt. O modelo aprende o padrão pelo exemplo e erra menos no formato. Copie um caso do gabarito e coloque antes da notícia a ser classificada.

### Dica 2: Peça ao modelo para explicar antes de responder (chain-of-thought)

Adicione algo como *"Antes de responder, explique brevemente seu raciocínio citando trechos da notícia."* Isso força o modelo a "pensar" antes de dar a resposta final — o notebook original usou exatamente essa técnica numa segunda iteração. Lembre-se de manter a seção de respostas exatas **depois** da explicação.

### Dica 3: Simplifique as instruções de formato

O prompt atual repete "escreva EXATAMENTE" muitas vezes. Experimente trocar por um bloco único com o template de saída esperado, por exemplo: `"Responda SOMENTE com as 4 linhas abaixo, preenchendo os colchetes:"` seguido do modelo.

### Dica 4: Mude o papel do sistema (system prompt)

O system prompt atual diz *"Você é um economista brasileiro..."*. Experimente papéis mais focados, como *"Você é um classificador de notícias que responde SOMENTE no formato solicitado"* — isso pode reduzir respostas fora do padrão.

### Dica 5: Ajuste a temperatura

A função `call_api` aceita o parâmetro `temperatura`. O padrão é 0 (mais determinístico). Tente valores como 0.3 ou 0.7 para ver se o modelo se sai melhor com um pouco de variação — mas cuidado: temperaturas altas podem gerar formatos inesperados.

> **Estratégia recomendada:** mude UMA coisa de cada vez e compare as métricas. Assim você sabe exatamente o que causou a melhora (ou piora).

## 6.1 Seu Prompt v2

Modifique a função `construir_prompt_v2` abaixo. Ela é uma cópia exata do prompt do Módulo 1 — os trechos marcados com `# >>> MODIFIQUE AQUI <<<` são os pontos sugeridos para alteração. Após modificar, execute a célula para testar com 5 notícias.

In [ ]:
# ============================================================
# Seu Prompt v2 — modifique e experimente!
# ============================================================

def construir_prompt_v2(noticia: str) -> str:
    """
    Versao modificada do prompt de classificacao.
    Altere os trechos marcados com '>>> MODIFIQUE AQUI <<<'
    e execute esta celula para testar suas mudancas.
    """

    # >>> MODIFIQUE AQUI <<< — Secao 1: Definicao da tarefa e perguntas
    # Dica: voce pode adicionar exemplos (few-shot) antes das perguntas
    #       ou pedir ao modelo para explicar o raciocinio (chain-of-thought).
    prompt = (
        "Leia e avalie a seguinte notícia sob as seguintes métricas:\n"
        "1. A notícia é internacional (fala sobre países além do Brasil) ou nacional?\n"
        "2. Ela demonstra incerteza sobre a economia do país ou países aos quais ela referencia?\n"
        "3. Se a notícia fala de um país além do Brasil e demonstra incerteza sobre a economia "
        "desse país, ela diz que essa incerteza afeta a economia brasileira negativamente?\n"
        "4. Essa notícia tem tom positivo, negativo, ou neutro?\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Exemplo de chain-of-thought (descomente as 3 linhas abaixo):
        # "Antes de responder, explique brevemente seu raciocínio, "
        # "citando trechos da notícia que justifiquem cada resposta. "
        # "Depois, apresente suas respostas na seção RESPOSTAS EXATAS abaixo.\n\n"

        # --- A noticia (NAO modifique esta linha) ---
        f"{noticia}\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Instrucoes de formato
        "É de extrema importância que você responda as quatro perguntas seguindo "
        "EXATAMENTE o padrão que lhe detalharei, pois suas respostas serão lidas por "
        "um programa de Python, que precisa ler trechos exatos para funcionar corretamente.\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Pergunta 1
        "Primeiro, se a notícia é nacional escreva EXATAMENTE: '1. A notícia é nacional'.\n"
        "Se a notícia é internacional, escreva EXATAMENTE: '1. A notícia é internacional'.\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Pergunta 2
        "Sobre a incerteza sobre a economia do país, se a notícia demonstra incerteza, "
        "escreva EXATAMENTE: "
        "'2. Sim, a notícia demonstra incerteza sobre a economia dos países analisados'.\n"
        "Se a notícia não demonstra incerteza, escreva EXATAMENTE: "
        "'2. Não, a notícia não demonstra incerteza sobre a economia dos países analisados'.\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Pergunta 3
        "Caso a notícia envolva países além do Brasil e também envolva incerteza, "
        "e essa incerteza afeta negativamente a economia brasileira, "
        "escreva EXATAMENTE: '3. A incerteza envolve negativamente a economia brasileira'.\n"
        "Se a incerteza não afeta a economia brasileira, escreva EXATAMENTE: "
        "'3. A incerteza não envolve negativamente a economia brasileira'.\n"
        "Se a questão sobre a incerteza econômica não se aplica, escreva EXATAMENTE: "
        "'3. A questão 3 não se aplica'.\n"
        "\n"

        # >>> MODIFIQUE AQUI <<< — Pergunta 4
        "Por fim, sobre o tom da notícia, se possui tom positivo, escreva EXATAMENTE: "
        "'4. A notícia possui tom positivo'.\n"
        "Se possui tom negativo, escreva EXATAMENTE: '4. A notícia possui tom negativo'.\n"
        "Se possui tom neutro, escreva EXATAMENTE: '4. A notícia possui tom neutro'."
    )

    return prompt


# ============================================================
# Teste rapido com 5 noticias usando seu prompt v2
# ============================================================

resultados_v2 = []

print("Testando seu prompt v2 com 5 noticias...")
print("=" * 50)

for indice, linha in df_noticias.head(5).iterrows():
    noticia = linha["noticia"]

    prompt = construir_prompt_v2(noticia)

    # >>> MODIFIQUE AQUI <<< — Experimente mudar a temperatura (ex: 0.3)
    print(f"\nNoticia {indice + 1}/5 - Enviando a API...", end=" ")
    resposta = call_api(prompt, temperatura=0)

    if resposta.startswith("ERRO:"):
        print(f"FALHA na API!")
        print(f"  Resposta: {resposta}")

    classificacoes = extrair_classificacoes(resposta)

    erros = [chave for chave, valor in classificacoes.items() if valor == "erro"]
    if erros and not resposta.startswith("ERRO:"):
        print(f"Extracao parcial - campos com erro: {erros}")
    elif not resposta.startswith("ERRO:"):
        print("OK")

    classificacoes["indice_original"] = indice
    classificacoes["resposta_api_bruta"] = resposta
    resultados_v2.append(classificacoes)

    time.sleep(1)

print("\n" + "=" * 50)
print(f"Teste concluido! {len(resultados_v2)} noticias classificadas com prompt v2.")

df_v2_teste = pd.DataFrame(resultados_v2)
colunas_exibir = ["indice_original", "internacional", "incerteza", "inc_afeta_br", "polaridade"]
print("\nResultados do seu prompt v2:")
print(df_v2_teste[colunas_exibir].to_string(index=False))

In [ ]:
# ============================================================
# Metricas do seu prompt v2
# ============================================================

resumo_v2 = calcular_metricas(
    df_gabarito=df_noticias,
    resultados=resultados_v2,
    colunas_avaliar=['incerteza', 'polaridade', 'internacional']
)

print("\nResumo armazenado na variavel 'resumo_v2'.")
print("Execute a proxima celula para comparar com o baseline do Modulo 1.")

## 6.2 Comparação: Baseline vs. Seu Prompt

A tabela abaixo compara as métricas do prompt original (Módulo 1) com as do seu prompt v2. Para cada dimensão, você verá a acurácia, a precisão ponderada e a quantidade de artigos avaliados.

Se alguma métrica melhorou, parabéns! Se piorou, volte à célula 6.1 e tente outra abordagem. Lembre-se: mude uma coisa de cada vez.

In [ ]:
# ============================================================
# Comparacao lado a lado: Modulo 1 (baseline) vs. Prompt v2
# ============================================================

if 'resumo_v2' not in dir() or not resumo_v2:
    print("=" * 65)
    print("  ATENCAO: 'resumo_v2' ainda nao foi gerado.")
    print("  Execute as celulas 6.1 e a celula de metricas acima primeiro.")
    print("=" * 65)
else:
    dimensoes = ['incerteza', 'polaridade', 'internacional']

    print("=" * 65)
    print(f"  {'DIMENSAO':<16s} | {'ACURACIA':^21s} | {'PRECISAO':^21s}")
    print(f"  {'':<16s} | {'Baseline':>9s}  {'v2':>9s} | {'Baseline':>9s}  {'v2':>9s}")
    print("-" * 65)

    for dim in dimensoes:
        base = resumo_modulo1.get(dim, {})
        acc_base = base.get('acuracia')
        prec_base = base.get('precisao')

        v2 = resumo_v2.get(dim, {})
        acc_v2 = v2.get('acuracia')
        prec_v2 = v2.get('precisao')

        acc_base_str = f"{acc_base:.1%}" if acc_base is not None else "  N/A  "
        acc_v2_str   = f"{acc_v2:.1%}" if acc_v2 is not None else "  N/A  "
        prec_base_str = f"{prec_base:.1%}" if prec_base is not None else "  N/A  "
        prec_v2_str   = f"{prec_v2:.1%}" if prec_v2 is not None else "  N/A  "

        def indicador(novo, antigo):
            if novo is None or antigo is None:
                return " "
            if novo > antigo + 0.005:
                return "+"
            elif novo < antigo - 0.005:
                return "-"
            return "="

        ind_acc = indicador(acc_v2, acc_base)
        ind_prec = indicador(prec_v2, prec_base)

        print(
            f"  {dim:<16s} | {acc_base_str:>9s}  {acc_v2_str:>9s} {ind_acc}"
            f"| {prec_base_str:>9s}  {prec_v2_str:>9s} {ind_prec}"
        )

    print("-" * 65)
    print("  Legenda: (+) melhorou  (-) piorou  (=) sem mudanca significativa")
    print("=" * 65)

### Suas Anotações

Use este espaço para documentar o que você mudou e por quê. Isso ajuda a manter um registro das suas iterações.

**O que eu mudei no prompt v2:**
- (descreva aqui a principal alteração que você fez)

**Resultado observado:**
- (a acurácia melhorou/piorou em quais dimensões?)

**Próxima iteração — o que eu tentaria:**
- (qual seria sua próxima ideia?)

---

# 7. Framework WIP — OpenAI Moderna (Responses API)

> **WORK IN PROGRESS — NAO TESTADO**
>
> Este módulo foi montado com base na documentação oficial da OpenAI (consultada em 24/03/2026),
> mas **não foi testado com chamadas reais à API**.
> Se encontrar erros, consulte os links de documentação no final desta seção.
> **Sua contribuição para testar e corrigir este módulo é um dos desafios deste notebook!**

---

### Por que uma nova API?

No **Módulo 1**, usamos a API de **Chat Completions** (`client.chat.completions.create()`) com o modelo `gpt-3.5-turbo`. O modelo retornava texto livre e nós precisávamos de **regex** para extrair as classificações — um processo frágil, sujeito a erros de parsing quando o modelo mudava ligeiramente o formato da resposta.

A partir de agosto de 2024, a OpenAI introduziu os **Structured Outputs** (saídas estruturadas), e mais recentemente a **Responses API**, que substitui a Chat Completions como a API recomendada para novos projetos. A grande mudança: agora podemos **forçar** o modelo a retornar dados exatamente no formato que definimos, eliminando completamente a necessidade de regex.

### O que mudou? Chat Completions vs. Responses API

| Aspecto | Chat Completions (Módulo 1) | Responses API (este módulo) |
|---|---|---|
| **Endpoint** | `client.chat.completions.create()` | `client.responses.parse()` |
| **Formato de saída** | Texto livre (precisa de regex/parsing) | JSON estruturado via schema |
| **Definição do schema** | Manual (instruções no prompt) | Automática via modelo **Pydantic** |
| **Garantia de formato** | Nenhuma — o modelo pode "inventar" | Total — a API **rejeita** saídas fora do schema |
| **Modelos compatíveis** | Todos (incluindo gpt-3.5-turbo) | `gpt-4o-2024-08-06`+, `gpt-4o-mini`, `gpt-5.x` |
| **Parâmetro de formato** | `response_format={"type": "json_object"}` | `text={"format": {"type": "json_schema", ...}}` |

> **Nota:** O `gpt-3.5-turbo` **não** suporta `json_schema` — apenas o modo JSON básico (sem garantia de schema). Por isso este módulo usa modelos mais recentes.

### O que é Pydantic e por que usamos?

[**Pydantic**](https://docs.pydantic.dev/) é a biblioteca padrão em Python para **validação de dados** usando type hints. Em vez de escrever um JSON Schema manualmente (trabalhoso e propenso a erros), definimos uma classe Python simples e o SDK da OpenAI converte automaticamente para o schema que a API precisa.

```
Classe Pydantic  -->  SDK converte para JSON Schema  -->  API força o modelo a seguir  -->  Resposta já chega parseada
```

O resultado: **zero regex, zero parsing manual, zero surpresas no formato**.

### Mesma API key

Este módulo usa a mesma variável `OPENAI_API_KEY` que já configuramos no início do notebook. Não é necessário configurar nada adicional.

### Links de referência

- [Guia de Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Guia de migração para Responses API](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [Modelos e deprecações](https://developers.openai.com/api/docs/deprecations)
- [Documentação do Pydantic](https://docs.pydantic.dev/)

## 7.1 Definindo o Schema com Pydantic

Antes de chamar a API, precisamos definir **exatamente** qual estrutura queremos na resposta. Fazemos isso criando um modelo Pydantic — uma classe Python que descreve cada campo, seu tipo e uma descrição.

Quando passamos esse modelo para `client.responses.parse()`, o SDK da OpenAI:
1. Converte a classe para um **JSON Schema**
2. Envia o schema junto com o prompt para a API
3. O modelo é **forçado** a retornar dados nesse formato exato
4. A resposta já volta como uma **instância da classe**, pronta para uso

Os 4 campos correspondem às dimensões de classificação do Módulo 1:

| Campo | Tipo | Descrição |
|---|---|---|
| `internacional` | `bool` | A notícia trata de tema internacional? |
| `incerteza` | `bool` | O texto demonstra incerteza econômica? |
| `inc_afeta_br` | `Optional[bool]` | Se internacional, a incerteza afeta o Brasil? (`None` quando N/A) |
| `polaridade` | `Literal["positivo", "negativo", "neutro"]` | Tom geral da notícia |

> **Dica:** Cada campo possui um `Field(description=...)`. Essas descrições são enviadas à API como parte do schema e ajudam o modelo a entender o que cada campo significa. Quanto mais claras as descrições, melhores as classificações!

In [ ]:
# =============================================================================
# 7.1 — Definição do Schema Pydantic para classificação de notícias
# =============================================================================
# Este schema define a estrutura EXATA que a API deve retornar.
# O modelo não pode inventar campos nem mudar tipos — é tudo ou nada.
# =============================================================================

from __future__ import annotations

from typing import Literal, Optional

from pydantic import BaseModel, Field


class ClassificacaoNoticia(BaseModel):
    """Schema de classificação de uma notícia econômica brasileira.

    Cada campo corresponde a uma dimensão de análise usada pelo IBRE/FGV.
    As descrições (Field description) são enviadas à API e orientam o modelo.
    """

    internacional: bool = Field(
        description=(
            "True se a notícia trata predominantemente de um tema internacional "
            "(ex.: economia de outros países, comércio exterior, geopolítica). "
            "False se o tema é predominantemente nacional/doméstico."
        )
    )

    incerteza: bool = Field(
        description=(
            "True se o texto demonstra ou menciona incerteza econômica "
            "(ex.: dúvidas sobre políticas, instabilidade, riscos, cenários indefinidos). "
            "False se o texto transmite informação sem tom de incerteza."
        )
    )

    inc_afeta_br: Optional[bool] = Field(
        default=None,
        description=(
            "Relevante APENAS quando 'internacional' é True E 'incerteza' é True. "
            "Nesse caso: True se a incerteza internacional afeta ou pode afetar o Brasil, "
            "False se não afeta o Brasil. "
            "Deve ser None (nulo) em todos os outros casos."
        ),
    )

    polaridade: Literal["positivo", "negativo", "neutro"] = Field(
        description=(
            "Tom geral da notícia em relação à economia: "
            "'positivo' para notícias otimistas/favoráveis, "
            "'negativo' para notícias pessimistas/desfavoráveis, "
            "'neutro' para notícias informativas sem viés claro."
        )
    )

    # >>> PERSONALIZAVEL <<<
    # Adicione novos campos aqui se quiser expandir a classificação!
    # Exemplo:
    # setor: Optional[str] = Field(
    #     default=None,
    #     description="Setor econômico principal da notícia (ex.: 'agronegócio', 'indústria')."
    # )


# ---------------------------------------------------------------------------
# Teste: verificar que o schema compila e aceita valores válidos
# ---------------------------------------------------------------------------

# Exemplo 1: notícia nacional, com incerteza, tom negativo
exemplo_1 = ClassificacaoNoticia(
    internacional=False,
    incerteza=True,
    inc_afeta_br=None,
    polaridade="negativo",
)

# Exemplo 2: notícia internacional, com incerteza que afeta o Brasil, tom neutro
exemplo_2 = ClassificacaoNoticia(
    internacional=True,
    incerteza=True,
    inc_afeta_br=True,
    polaridade="neutro",
)

print("Schema ClassificacaoNoticia compilou com sucesso!")
print(f"\nExemplo 1 (nacional, incerteza, negativo):")
print(exemplo_1.model_dump_json(indent=2))
print(f"\nExemplo 2 (internacional, incerteza afeta BR, neutro):")
print(exemplo_2.model_dump_json(indent=2))

# Mostrar o JSON Schema que será enviado à API
print(f"\nJSON Schema gerado automaticamente:")
import json as _json
print(_json.dumps(ClassificacaoNoticia.model_json_schema(), indent=2, ensure_ascii=False))

---

## 7.2 Classificação com Responses API

> **WORK IN PROGRESS — CÓDIGO NÃO TESTADO COM CHAMADAS REAIS**
>
> A função abaixo foi escrita com base na documentação oficial da OpenAI (consultada em 24/03/2026).
> Os nomes de métodos e parâmetros podem mudar — se algo não funcionar, consulte os links no final desta seção.

### A grande simplificação

Compare com o Módulo 1:

| Passo | Módulo 1 (Chat Completions) | Este módulo (Responses API) |
|-------|-----------------------------|-----------------------------|
| **1. Prompt** | Instruções detalhadas de formato + frases exatas | Mesmo `SYSTEM_PROMPT` — o schema cuida do formato |
| **2. Chamada** | `client.chat.completions.create()` | `client.responses.parse()` |
| **3. Parsing** | Regex frágil (`re.search(...)`) | Automático — `.output_parsed` já é um objeto Pydantic |
| **4. Resultado** | `dict` com strings e possíveis `"erro"` | Instância de `ClassificacaoNoticia` com tipos corretos |

O prompt pode ser **muito mais simples** porque não precisamos mais pedir formato rígido — o schema Pydantic garante a estrutura.

In [ ]:
# =============================================================================
# 7.2 — Classificacao com Responses API (Structured Outputs)
# =============================================================================
# WORK IN PROGRESS — Baseado na documentacao da OpenAI de 24/03/2026.
# Este codigo usa client.responses.parse() com o schema Pydantic definido
# na celula 7.1. A resposta ja volta parseada — sem regex!
# =============================================================================

import time


def classificar_com_responses_api(
    noticia: str,
    modelo: str = "gpt-4o-mini",
):
    """
    Classifica uma noticia usando a Responses API com Structured Outputs.

    Diferente do Modulo 1, esta funcao:
    - Nao precisa de regex para extrair resultados
    - Retorna diretamente uma instancia de ClassificacaoNoticia
    - O schema Pydantic GARANTE que a resposta tera o formato correto

    Parametros
    ----------
    noticia : str
        O texto da noticia a ser classificada.
    modelo : str, opcional
        Modelo da OpenAI. Padrao: "gpt-4o-mini" (mais barato).
        Alternativas: "gpt-4o", modelos gpt-5.x.
        IMPORTANTE: gpt-3.5-turbo NAO e compativel com Structured Outputs.

    Retorna
    -------
    ClassificacaoNoticia ou None
        Instancia com os campos classificados, ou None em caso de erro.
    """

    # O prompt pode ser mais simples que no Modulo 1: nao precisamos pedir
    # formato rigido nem frases exatas. O schema Pydantic cuida de tudo.
    prompt_usuario = (
        "Classifique a seguinte notícia econômica brasileira nas dimensões "
        "definidas no schema de resposta. Analise cuidadosamente o conteúdo "
        "antes de responder.\n\n"
        f"Notícia:\n{noticia}"
    )

    try:
        resposta = cliente_openai.responses.parse(
            model=modelo,
            input=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt_usuario},
            ],
            text_format=ClassificacaoNoticia,
        )

        resultado = resposta.output_parsed
        return resultado

    except Exception as erro:
        tipo_erro = type(erro).__name__

        if "AuthenticationError" in tipo_erro:
            print("ERRO DE AUTENTICACAO:")
            print("  Sua chave de API esta invalida ou expirada.")
            print(f"  Detalhe: {erro}")

        elif "RateLimitError" in tipo_erro:
            print("ERRO DE LIMITE DE REQUISICOES (Rate Limit):")
            print("  Aguarde alguns segundos e tente novamente.")
            print(f"  Detalhe: {erro}")

        elif "NotFoundError" in tipo_erro or "model" in str(erro).lower():
            print("ERRO DE MODELO NAO ENCONTRADO:")
            print(f"  O modelo '{modelo}' pode nao estar disponivel.")
            print("  Tente 'gpt-4o-mini' ou consulte:")
            print("  https://developers.openai.com/api/docs/deprecations")
            print(f"  Detalhe: {erro}")

        elif "ValidationError" in tipo_erro or "pydantic" in str(erro).lower():
            print("ERRO DE VALIDACAO PYDANTIC:")
            print("  A resposta nao correspondeu ao schema ClassificacaoNoticia.")
            print("  Consulte: https://developers.openai.com/api/docs/guides/structured-outputs")
            print(f"  Detalhe: {erro}")

        else:
            print(f"ERRO INESPERADO ({tipo_erro}):")
            print(f"  {erro}")
            print("  Verifique: https://status.openai.com/")

        return None


print("Funcao 'classificar_com_responses_api' definida.")
print(f"  Modelo padrao: gpt-4o-mini")
print(f"  Schema: ClassificacaoNoticia")


# =============================================================================
# Teste rapido — 5 noticias com a Responses API
# =============================================================================

EXECUTAR_MODULO_2 = False  # <-- Mude para True para executar

if EXECUTAR_MODULO_2:

    resultados_modulo2 = []
    n_teste = 5

    print(f"\nIniciando teste com {n_teste} noticias via Responses API...")
    print("=" * 60)

    for indice, linha in df_noticias.head(n_teste).iterrows():
        noticia = linha["noticia"]

        print(f"[{indice + 1}/{n_teste}] Classificando...", end=" ")
        resultado = classificar_com_responses_api(noticia)

        if resultado is not None:
            resultado_dict = {
                "internacional": int(resultado.internacional),
                "incerteza": int(resultado.incerteza),
                "inc_afeta_br": (
                    int(resultado.inc_afeta_br)
                    if resultado.inc_afeta_br is not None
                    else None
                ),
                "polaridade": {
                    "positivo": 1, "negativo": -1, "neutro": 0,
                }[resultado.polaridade],
                "indice_original": indice,
            }
            print("OK")
            print(f"         -> {resultado.model_dump()}")
            resultados_modulo2.append(resultado_dict)
        else:
            resultados_modulo2.append({
                "internacional": "erro", "incerteza": "erro",
                "inc_afeta_br": "erro", "polaridade": "erro",
                "indice_original": indice,
            })
            print("FALHA (veja mensagem acima)")

        time.sleep(1)

    print("\n" + "=" * 60)
    print(f"Teste concluido! {len(resultados_modulo2)} noticias processadas.")

    df_modulo2 = pd.DataFrame(resultados_modulo2)
    colunas_exibir = ["indice_original", "internacional", "incerteza", "inc_afeta_br", "polaridade"]
    print("\nResultados do Modulo 2 (Responses API):")
    print(df_modulo2[colunas_exibir].to_string(index=False))

    total_erros = sum(
        1 for r in resultados_modulo2
        for k in ["internacional", "incerteza", "inc_afeta_br", "polaridade"]
        if r[k] == "erro"
    )
    print(f"\nTotal de erros: {total_erros} de {len(resultados_modulo2) * 4} campos")

    if total_erros == 0:
        print("Nenhum erro! A Responses API retornou dados estruturados para todas as noticias.")

    noticias_ok = [r for r in resultados_modulo2 if r["incerteza"] != "erro"]
    if noticias_ok:
        print("\n--- Metricas do Modulo 2 ---")
        resumo_modulo2 = calcular_metricas(
            df_gabarito=df_noticias,
            resultados=noticias_ok,
            colunas_avaliar=["incerteza", "polaridade", "internacional"],
        )

else:
    resultados_modulo2 = []
    print("Modulo 2 (Responses API) desativado.")
    print("  Para executar, altere EXECUTAR_MODULO_2 para True acima.")
    print("  DICA: Este modulo e o Desafio 1 do notebook!")

## 7.3 Avaliação de Métricas — Módulo 2

Se você executou o módulo acima, vamos calcular as mesmas métricas que usamos no Módulo 1 (acurácia, precisão, matriz de confusão). Isso permite comparar diretamente o desempenho da **Responses API** com o **GPT-3.5-Turbo + regex**.

In [ ]:
# ============================================================
# 7.3 — Métricas do Módulo 2 (Responses API)
# ============================================================

if 'resultados_modulo2' in dir() and resultados_modulo2:
    print("Calculando métricas do Módulo 2 (Responses API)...")
    print()
    resumo_modulo2 = calcular_metricas(
        df_gabarito=df_noticias,
        resultados=resultados_modulo2,
        colunas_avaliar=['incerteza', 'polaridade', 'internacional']
    )
    print()
    print("Resultados salvos em 'resumo_modulo2'.")
else:
    resumo_modulo2 = None
    print("Módulo 2 não foi executado — pulando métricas.")
    print("(Rode a célula 7.2 primeiro se quiser avaliar a Responses API.)")

---

> **ATENCAO: MODULO EM CONSTRUCAO (WIP)**
>
> **Todo o código da seção 7 (Responses API) foi escrito com base na documentação da OpenAI
> consultada em 24/03/2026, mas NÃO FOI TESTADO com chamadas reais à API.**
>
> A API da OpenAI evolui rapidamente. Nomes de métodos, parâmetros e comportamentos
> podem ter mudado desde a data de consulta. **Esperamos que haja erros** — e corrigi-los
> faz parte do aprendizado!

---

### Problemas comuns e soluções

| Problema | Sintoma | Solução |
|----------|---------|---------|
| **Modelo não encontrado** | `NotFoundError` | Verifique o nome do modelo. Tente `gpt-4o-mini`. Consulte a [página de modelos](https://developers.openai.com/api/docs/deprecations). |
| **Erro de autenticação** | `AuthenticationError` | Verifique se `OPENAI_API_KEY` está correta. A chave deve começar com `sk-`. |
| **Erro de validação Pydantic** | `ValidationError` | O schema pode não corresponder ao que a API retorna. Consulte a [documentação de Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs). |
| **Rate limit** | `RateLimitError` | Aguarde 30-60 segundos e tente novamente. |
| **Método não existe** | `AttributeError: 'OpenAI' has no attribute 'responses'` | SDK desatualizado. Execute `pip install --upgrade openai` e reinicie o kernel. |
| **Parâmetro não reconhecido** | `TypeError: unexpected keyword argument 'text_format'` | O nome do parâmetro pode ter mudado. Consulte o [guia de migração](https://developers.openai.com/api/docs/guides/migrate-to-responses). |

### Links de referência

- [Guia de Structured Outputs](https://developers.openai.com/api/docs/guides/structured-outputs)
- [Guia de migração para Responses API](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- [Modelos e deprecações](https://developers.openai.com/api/docs/deprecations)
- [Documentação do Pydantic](https://docs.pydantic.dev/)
- [Status da API OpenAI](https://status.openai.com/)

---

### Desafio 1: Faça este módulo funcionar!

Este módulo é o **primeiro desafio** deste notebook. Seu objetivo:

1. **Execute** a célula acima com `EXECUTAR_MODULO_2 = True`
2. **Corrija** os erros que aparecerem (use os links acima como referência)
3. **Compare** os resultados com o Módulo 1 — as métricas melhoraram? Os erros de extração desapareceram?
4. **Reflita**: mesmo que as métricas de classificação sejam similares, o que mudou em termos de **robustez** do pipeline?

> **Dica:** Se conseguir fazer o módulo funcionar, tente também com `modelo="gpt-4o"` e compare se um modelo maior produz classificações mais precisas. Documente suas descobertas!

---

# 8. Framework WIP — Google Gemini

> **WORK IN PROGRESS — NAO TESTADO**
>
> Este módulo foi montado com base na documentação oficial do Google Gemini (consultada em 24/03/2026),
> mas **NÃO FOI TESTADO com chamadas reais à API**.
> Se encontrar erros, consulte os links de documentação no final desta seção.
> **Sua contribuição para testar e corrigir este módulo é o Desafio 2 deste notebook!**

---

### O que é o Google Gemini?

**Gemini** é a família de modelos de linguagem do Google. Assim como o GPT da OpenAI, os modelos Gemini podem classificar textos, responder perguntas e gerar conteúdo — mas com uma diferença crucial para este projeto: **a API tem um tier gratuito generoso**.

### Por que Gemini importa neste notebook?

| Aspecto | OpenAI (Módulos 1 e 2) | Google Gemini (este módulo) |
|---------|------------------------|-----------------------------|
| **Chave de API** | Fornecida pelo(a) instrutor(a) | **Sua própria chave gratuita** |
| **Custo** | Pago (custo por token) | **Gratuito** dentro dos limites |
| **Cadastro** | Precisa de conta OpenAI com créditos | Apenas conta Google pessoal |
| **Acesso pós-curso** | Depende da chave do(a) instrutor(a) | **Você mantém acesso permanente** |

Isso significa que, após o curso, você pode continuar experimentando com classificação de notícias **sem custo algum**, usando sua própria chave Gemini.

### Como obter sua chave de API (já feito no Setup)

Se você ainda não criou sua chave, siga estes passos:

1. Acesse [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)
2. Faça login com sua conta Google
3. Clique em **"Create API Key"** (Criar chave de API)
4. Escolha um projeto do Google Cloud (ou crie um novo — é gratuito)
5. Copie a chave gerada e guarde em local seguro

A chave já foi carregada na variável `GOOGLE_API_KEY` na seção de Setup. Se você configurou corretamente, não precisa fazer mais nada aqui.

### Limites do tier gratuito

O tier gratuito do Gemini é generoso, mas tem limites:

- **Requisições por minuto (RPM):** variam por modelo (tipicamente 10-15 RPM para modelos Flash)
- **Tokens por minuto (TPM):** limite de tokens enviados + recebidos por minuto
- **Requisições por dia (RPD):** limite diário de chamadas

Para nosso caso de uso (dezenas de notícias curtas), esses limites são mais que suficientes. Consulte os limites atualizados em: [ai.google.dev/gemini-api/docs/pricing](https://ai.google.dev/gemini-api/docs/pricing)

### SDK utilizado: `google-genai`

> **Atenção:** O Google tem **dois** SDKs Python para Gemini:
> - `google-generativeai` — SDK antigo, **descontinuado** (deprecated)
> - `google-genai` — SDK novo e atual, que usamos neste notebook
>
> Se você encontrar tutoriais usando `import google.generativeai as genai`, saiba que esse é o SDK antigo.
> Nosso código usa `from google import genai`, que é o SDK recomendado.

### Structured Outputs no Gemini

Assim como a OpenAI (Módulo 2), o Gemini também suporta **saídas estruturadas**. A diferença é que o Gemini usa **JSON Schema puro** (um dicionário Python), enquanto a OpenAI aceita classes Pydantic diretamente.

A boa notícia: podemos **reutilizar** o modelo Pydantic `ClassificacaoNoticia` do Módulo 2! Basta chamar `.model_json_schema()` para gerar o JSON Schema equivalente.

### Links de referência

- [Documentação do google-genai](https://googleapis.github.io/python-genai/)
- [Guia de Structured Output do Gemini](https://ai.google.dev/gemini-api/docs/structured-output)
- [Preços e limites gratuitos](https://ai.google.dev/gemini-api/docs/pricing)
- [Criar chave de API](https://aistudio.google.com/app/apikey)
- [Modelos disponíveis](https://ai.google.dev/gemini-api/docs/models)

## 8.1 Schema e Configuração

No Módulo 2, definimos o schema `ClassificacaoNoticia` como uma classe Pydantic. A OpenAI aceita essa classe diretamente — o SDK converte para JSON Schema internamente.

O Gemini funciona de forma ligeiramente diferente: ele precisa receber o **JSON Schema como um dicionário Python**, não a classe Pydantic em si. Mas isso não é problema — o Pydantic tem o método `.model_json_schema()` que gera exatamente o que precisamos.

### O que o código abaixo faz

1. **Importa** o SDK `google-genai` e cria o cliente com sua chave
2. **Gera** o JSON Schema a partir do modelo Pydantic (reutilizando `ClassificacaoNoticia`)
3. **Define** a configuração (`config`) que será passada em cada chamada à API

### Sobre o `config`

| Chave | Valor | Função |
|-------|-------|--------|
| `response_mime_type` | `"application/json"` | Força a resposta a ser JSON válido |
| `response_json_schema` | `dict` (JSON Schema) | Define a estrutura exata do JSON |

In [ ]:
# =============================================================================
# 8.1 — Schema e Configuracao para o Google Gemini
# =============================================================================
# WORK IN PROGRESS — Baseado na documentacao do Google Gemini de 24/03/2026.
# Reutilizamos o modelo Pydantic ClassificacaoNoticia (definido na celula 7.1)
# e convertemos para JSON Schema, que e o formato aceito pelo Gemini.
# =============================================================================

from google import genai

# --- Cliente Gemini ---
# Usa a chave GOOGLE_API_KEY carregada no Setup
cliente_gemini = genai.Client(api_key=GOOGLE_API_KEY)

# --- Gerar JSON Schema a partir do modelo Pydantic ---
# ClassificacaoNoticia foi definida na celula 7.1 (Modulo 2).
schema_classificacao = ClassificacaoNoticia.model_json_schema()

# --- Configuracao de Structured Output para o Gemini ---
config_gemini = {
    "response_mime_type": "application/json",
    "response_json_schema": schema_classificacao,
}

# --- Verificacao ---
print("Cliente Gemini criado com sucesso.")
print()
print("JSON Schema gerado a partir de ClassificacaoNoticia:")
print("-" * 50)

import json as _json
print(_json.dumps(schema_classificacao, indent=2, ensure_ascii=False))
print("-" * 50)
print()
print("Configuracao completa (config_gemini):")
print(f"  response_mime_type: {config_gemini['response_mime_type']}")
print(f"  response_json_schema: {list(schema_classificacao.keys())}")

## 8.2 Classificação com Gemini

> **WORK IN PROGRESS — CÓDIGO NÃO TESTADO COM CHAMADAS REAIS**

### Diferença na chamada

| Aspecto | OpenAI (Módulo 2) | Gemini (este módulo) |
|---------|--------------------|-----------------------|
| **Método** | `client.responses.parse()` | `client.models.generate_content()` |
| **Prompt** | Lista de mensagens `[{"role": ..., "content": ...}]` | String única (system + user concatenados) |
| **Schema** | Classe Pydantic diretamente | JSON Schema via `config` dict |
| **Resposta** | `.output_parsed` (objeto Pydantic) | `.text` (string JSON) + `json.loads()` |

Note que o Gemini retorna o JSON como **texto** — precisamos de `json.loads()` para converter em dicionário Python. Isso é uma etapa a mais comparado com a OpenAI, mas ainda muito mais robusto que o regex do Módulo 1.

In [ ]:
# =============================================================================
# 8.2 — Classificacao com Gemini (Structured Output)
# =============================================================================
# WORK IN PROGRESS — Baseado na documentacao do Google Gemini de 24/03/2026.
# =============================================================================

import json
import time


def classificar_com_gemini(
    noticia: str,
    modelo: str = "gemini-2.5-flash",
):
    """
    Classifica uma noticia usando o Google Gemini com Structured Output.

    Parametros
    ----------
    noticia : str
        O texto da noticia a ser classificada.
    modelo : str, opcional
        Modelo Gemini. Padrao: "gemini-2.5-flash" (rapido e gratuito).
        Alternativas: "gemini-2.0-flash", "gemini-2.5-pro".

    Retorna
    -------
    dict ou None
        Dicionario com os campos classificados, ou None em caso de erro.
    """

    prompt_completo = (
        f"{SYSTEM_PROMPT}\n\n"
        "Classifique a seguinte notícia econômica brasileira nas dimensões "
        "definidas no schema de resposta. Analise cuidadosamente o conteúdo "
        "antes de responder.\n\n"
        f"Notícia:\n{noticia}"
    )

    try:
        resposta = cliente_gemini.models.generate_content(
            model=modelo,
            contents=prompt_completo,
            config=config_gemini,
        )

        resultado_dict = json.loads(resposta.text)
        return resultado_dict

    except json.JSONDecodeError as erro:
        print("ERRO DE PARSING JSON:")
        print("  A resposta do Gemini nao e um JSON valido.")
        texto = getattr(resposta, 'text', 'N/A') if 'resposta' in dir() else 'N/A'
        print(f"  Resposta recebida: {str(texto)[:200]}")
        print(f"  Detalhe: {erro}")
        return None

    except Exception as erro:
        tipo_erro = type(erro).__name__

        if "PermissionDenied" in tipo_erro or "403" in str(erro):
            print("ERRO DE PERMISSAO / AUTENTICACAO:")
            print("  Sua GOOGLE_API_KEY pode estar invalida.")
            print("  Crie uma nova em: https://aistudio.google.com/app/apikey")
            print(f"  Detalhe: {erro}")

        elif "ResourceExhausted" in tipo_erro or "429" in str(erro):
            print("ERRO DE LIMITE (Rate Limit):")
            print("  Aguarde 60 segundos e tente novamente.")
            print("  Limites: https://ai.google.dev/gemini-api/docs/pricing")
            print(f"  Detalhe: {erro}")

        elif "NotFound" in tipo_erro or "404" in str(erro):
            print("ERRO DE MODELO NAO ENCONTRADO:")
            print(f"  O modelo '{modelo}' pode nao estar disponivel.")
            print("  Tente 'gemini-2.0-flash' ou 'gemini-2.5-flash'.")
            print(f"  Detalhe: {erro}")

        else:
            print(f"ERRO INESPERADO ({tipo_erro}):")
            print(f"  {erro}")
            print("  Documentacao: https://googleapis.github.io/python-genai/")

        return None


print("Funcao 'classificar_com_gemini' definida.")
print(f"  Modelo padrao: gemini-2.5-flash")
print(f"  API: GRATUITA (dentro dos limites do free tier)")


# =============================================================================
# Teste rapido — 5 noticias com o Gemini
# =============================================================================

EXECUTAR_MODULO_3 = False  # <-- Mude para True para executar

if EXECUTAR_MODULO_3:

    resultados_modulo3 = []
    n_teste = 5

    print(f"\nIniciando teste com {n_teste} noticias via Gemini...")
    print("=" * 60)

    for indice, linha in df_noticias.head(n_teste).iterrows():
        noticia = linha["noticia"]

        print(f"[{indice + 1}/{n_teste}] Classificando...", end=" ")
        resultado = classificar_com_gemini(noticia)

        if resultado is not None:
            try:
                resultado_numerico = {
                    "internacional": int(resultado["internacional"]),
                    "incerteza": int(resultado["incerteza"]),
                    "inc_afeta_br": (
                        int(resultado["inc_afeta_br"])
                        if resultado.get("inc_afeta_br") is not None
                        else None
                    ),
                    "polaridade": {
                        "positivo": 1, "negativo": -1, "neutro": 0,
                    }[resultado["polaridade"]],
                    "indice_original": indice,
                }
                print("OK")
                print(f"         -> {resultado}")
                resultados_modulo3.append(resultado_numerico)
            except (KeyError, ValueError) as erro:
                print(f"ERRO ao converter: {erro}")
                resultados_modulo3.append({
                    "internacional": "erro", "incerteza": "erro",
                    "inc_afeta_br": "erro", "polaridade": "erro",
                    "indice_original": indice,
                })
        else:
            resultados_modulo3.append({
                "internacional": "erro", "incerteza": "erro",
                "inc_afeta_br": "erro", "polaridade": "erro",
                "indice_original": indice,
            })
            print("FALHA (veja mensagem acima)")

        time.sleep(2)  # Pausa maior para free tier

    print("\n" + "=" * 60)
    print(f"Teste concluido! {len(resultados_modulo3)} noticias processadas.")

    df_modulo3 = pd.DataFrame(resultados_modulo3)
    colunas_exibir = ["indice_original", "internacional", "incerteza", "inc_afeta_br", "polaridade"]
    print("\nResultados do Modulo 3 (Gemini):")
    print(df_modulo3[colunas_exibir].to_string(index=False))

    total_erros = sum(
        1 for r in resultados_modulo3
        for k in ["internacional", "incerteza", "inc_afeta_br", "polaridade"]
        if r[k] == "erro"
    )
    print(f"\nTotal de erros: {total_erros} de {len(resultados_modulo3) * 4} campos")

    noticias_ok = [r for r in resultados_modulo3 if r["incerteza"] != "erro"]
    if noticias_ok:
        print("\n--- Metricas do Modulo 3 (Gemini) ---")
        resumo_modulo3 = calcular_metricas(
            df_gabarito=df_noticias,
            resultados=noticias_ok,
            colunas_avaliar=["incerteza", "polaridade", "internacional"],
        )

else:
    resultados_modulo3 = []
    print("Modulo 3 (Gemini) desativado.")
    print("  Para executar, altere EXECUTAR_MODULO_3 para True acima.")
    print("  DICA: Este modulo e o Desafio 2 do notebook!")

## 8.3 Avaliação de Métricas — Módulo 3

Se você executou o módulo acima, vamos calcular as métricas para o **Google Gemini** usando a mesma função de avaliação.

In [ ]:
# ============================================================
# 8.3 — Métricas do Módulo 3 (Google Gemini)
# ============================================================

if 'resultados_modulo3' in dir() and resultados_modulo3:
    print("Calculando métricas do Módulo 3 (Google Gemini)...")
    print()
    resumo_modulo3 = calcular_metricas(
        df_gabarito=df_noticias,
        resultados=resultados_modulo3,
        colunas_avaliar=['incerteza', 'polaridade', 'internacional']
    )
    print()
    print("Resultados salvos em 'resumo_modulo3'.")
else:
    resumo_modulo3 = None
    print("Módulo 3 não foi executado — pulando métricas.")
    print("(Rode a célula 8.2 primeiro se quiser avaliar o Gemini.)")

## 8.3 Comparação: OpenAI vs. Gemini

A tabela abaixo compara as três abordagens que vimos neste notebook:

| Aspecto | Módulo 1: Chat Completions | Módulo 2: Responses API | Módulo 3: Gemini |
|---------|---------------------------|------------------------|------------------|
| **SDK Python** | `openai` | `openai` | `google-genai` |
| **Import** | `from openai import OpenAI` | `from openai import OpenAI` | `from google import genai` |
| **Autenticação** | `OPENAI_API_KEY` (instrutor) | `OPENAI_API_KEY` (instrutor) | `GOOGLE_API_KEY` (sua, gratuita) |
| **Método de chamada** | `client.chat.completions.create()` | `client.responses.parse()` | `client.models.generate_content()` |
| **Structured Output** | Não (regex manual) | Sim (Pydantic direto) | Sim (JSON Schema via config) |
| **Formato da resposta** | Texto livre | Objeto Pydantic | Texto JSON |
| **Parsing necessário** | Regex (`re.search`) | Nenhum (`.output_parsed`) | `json.loads()` |
| **Tier gratuito** | Não | Não | **Sim** (generoso) |
| **Modelos recomendados** | `gpt-3.5-turbo` | `gpt-4o-mini`, `gpt-4o` | `gemini-2.5-flash` |
| **Robustez do parsing** | Frágil (depende do formato) | Máxima (schema forçado) | Alta (JSON forçado) |

### Qual escolher?

- **Para aprender:** Módulo 1 — mostra o pipeline completo, incluindo as dificuldades de parsing
- **Para produção (OpenAI):** Módulo 2 — parsing robusto, sem regex, schema garantido
- **Para experimentar sem custo:** Módulo 3 — API gratuita, ótimo para projetos pessoais
- **Para comparar modelos:** Execute os 3 módulos e compare as métricas com `calcular_metricas()`

---

> **ATENÇÃO: MÓDULO EM CONSTRUÇÃO (WIP)**
>
> **Todo o código da seção 8 (Google Gemini) foi escrito com base na documentação oficial
> consultada em 24/03/2026, mas NÃO FOI TESTADO com chamadas reais à API.**
>
> A API do Gemini evolui rapidamente. Nomes de métodos, parâmetros e comportamentos
> podem ter mudado desde a data de consulta. **Esperamos que haja erros** — e corrigi-los
> faz parte do aprendizado!

---

### Problemas comuns e soluções

| Problema | Sintoma | Solução |
|----------|---------|---------|
| **SDK errado instalado** | `ModuleNotFoundError: No module named 'google.genai'` | Instale `google-genai` (NÃO `google-generativeai`). Execute `pip install google-genai` e reinicie o kernel. |
| **Chave inválida** | `PermissionDenied` / `403` | Verifique sua `GOOGLE_API_KEY`. Crie uma nova em [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey). |
| **Modelo não encontrado** | `NotFound` / `404` | Tente `gemini-2.0-flash`. Consulte os [modelos disponíveis](https://ai.google.dev/gemini-api/docs/models). |
| **Limite excedido** | `ResourceExhausted` / `429` | Aguarde 60 segundos. Consulte os [limites atuais](https://ai.google.dev/gemini-api/docs/pricing). |
| **JSON inválido** | `JSONDecodeError` | Tente novamente ou use um modelo maior (`gemini-2.5-pro`). |
| **Import antigo** | `import google.generativeai as genai` | SDK **antigo** (deprecated). Use `from google import genai`. |
| **Parâmetro desconhecido** | `TypeError: unexpected keyword argument` | Consulte a [documentação do SDK](https://googleapis.github.io/python-genai/). |

### Links de referência

- [Documentação do SDK google-genai](https://googleapis.github.io/python-genai/)
- [Guia de Structured Output do Gemini](https://ai.google.dev/gemini-api/docs/structured-output)
- [Preços e limites do tier gratuito](https://ai.google.dev/gemini-api/docs/pricing)
- [Modelos Gemini disponíveis](https://ai.google.dev/gemini-api/docs/models)
- [Criar chave de API](https://aistudio.google.com/app/apikey)

---

### Desafio 2: Faça este módulo funcionar!

Este módulo é o **segundo desafio** deste notebook. Seu objetivo:

1. **Configure** sua `GOOGLE_API_KEY` (se ainda não fez, volte à seção 3)
2. **Execute** a célula acima com `EXECUTAR_MODULO_3 = True`
3. **Corrija** os erros que aparecerem (use a tabela acima como guia)
4. **Compare** os resultados com os Módulos 1 e 2
5. **Experimente** com modelos diferentes: `gemini-2.0-flash` vs. `gemini-2.5-flash`
6. **Documente** suas descobertas!

> **Bônus:** Execute os 3 módulos com o dataset completo e compare as métricas lado a lado.

---

# 9. Comparação de Desempenho e Análise de Erros

Nesta seção, comparamos o desempenho dos modelos que você executou e analisamos as notícias onde o modelo divergiu da classificação humana.

**O que vamos fazer:**
1. **Tabela comparativa** — acurácia e precisão de cada provedor, lado a lado
2. **Lista de erros** — quais notícias foram misclassificadas e em qual dimensão
3. **Inspeção detalhada** — ver a notícia completa, a resposta bruta do modelo, e comparar com o gabarito humano

> **Nota:** A comparação inclui automaticamente só os módulos que você executou. Se rodou apenas o Módulo 1, verá só ele. Se rodou os 3, verá os 3 lado a lado.

In [ ]:
# ============================================================
# 9.1 — Funções de Comparação e Análise de Erros
# ============================================================

# Mapeamento de nomes amigáveis para variáveis de resultado
_MODULOS_DISPONIVEIS = {
    "GPT-3.5-Turbo (regex)": ("resumo_modulo1", "resultados_classificacao", "resultados_teste"),
    "OpenAI Responses API":  ("resumo_modulo2", "resultados_modulo2"),
    "Google Gemini":         ("resumo_modulo3", "resultados_modulo3"),
}


def _obter_resultados(nome_modulo):
    """Retorna (resumo, resultados_lista) para um módulo, ou (None, None) se não disponível."""
    info = _MODULOS_DISPONIVEIS.get(nome_modulo)
    if not info:
        return None, None
    resumo_var = info[0]
    resumo = globals().get(resumo_var)
    resultados = None
    for var in info[1:]:
        resultados = globals().get(var)
        if resultados:
            break
    return resumo, resultados


def comparar_provedores():
    """
    Compara acurácia e precisão entre todos os provedores que foram executados.
    Mostra uma tabela lado a lado.
    """
    disponiveis = {}
    for nome in _MODULOS_DISPONIVEIS:
        resumo, _ = _obter_resultados(nome)
        if resumo is not None:
            disponiveis[nome] = resumo

    if not disponiveis:
        print("Nenhum módulo foi executado ainda.")
        print("Execute pelo menos o Módulo 1 (seção 4) para ver métricas.")
        return

    dimensoes = ['incerteza', 'polaridade', 'internacional']
    col_w = 24

    print("=" * 70)
    print("COMPARAÇÃO DE DESEMPENHO ENTRE PROVEDORES")
    print("=" * 70)
    print()

    for dim in dimensoes:
        valores_acc = [d[dim]['acuracia'] for d in disponiveis.values() if dim in d]
        valores_prec = [d[dim]['precisao'] for d in disponiveis.values() if dim in d]
        melhor_acc = max(valores_acc) if valores_acc else 0
        melhor_prec = max(valores_prec) if valores_prec else 0

        print(f"━━━ {dim.upper()} ━━━")
        print(f"  {'Provedor':<{col_w}} {'Acurácia':>10} {'Precisão':>10} {'N válidos':>10}")
        print(f"  {'-' * col_w} {'-' * 10} {'-' * 10} {'-' * 10}")

        for nome, resumo in disponiveis.items():
            if dim not in resumo:
                continue
            m = resumo[dim]
            acc = m['acuracia']
            prec = m['precisao']
            n = m['n_validos']
            acc_mark = " *" if acc == melhor_acc and len(disponiveis) > 1 else ""
            prec_mark = " *" if prec == melhor_prec and len(disponiveis) > 1 else ""
            print(f"  {nome:<{col_w}} {acc:>9.1%}{acc_mark} {prec:>9.1%}{prec_mark} {n:>10d}")
        print()

    if len(disponiveis) > 1:
        print("  * = melhor valor nesta dimensão")
    print()
    print(f"Provedores comparados: {len(disponiveis)} de {len(_MODULOS_DISPONIVEIS)}")
    if len(disponiveis) < len(_MODULOS_DISPONIVEIS):
        faltam = [n for n in _MODULOS_DISPONIVEIS if n not in disponiveis]
        print(f"Não executados: {', '.join(faltam)}")


def _codificar_valor(valor):
    """Normaliza um valor de classificação para comparação."""
    if valor is None or (isinstance(valor, float) and str(valor) == 'nan'):
        return None
    if isinstance(valor, str):
        mapa = {"sim": 1, "não": 0, "nao": 0,
                "positivo": 1, "negativo": -1, "neutro": 0,
                "erro": "erro"}
        return mapa.get(valor.lower().strip(), valor)
    return valor


def listar_erros(modulo="GPT-3.5-Turbo (regex)"):
    """
    Lista todas as notícias onde o modelo divergiu da classificação humana.

    Parâmetros:
        modulo (str): Nome do módulo. Opções:
            - "GPT-3.5-Turbo (regex)"  (padrão)
            - "OpenAI Responses API"
            - "Google Gemini"
    """
    _, resultados = _obter_resultados(modulo)
    if not resultados:
        print(f"Módulo '{modulo}' não foi executado.")
        return []

    dimensoes = ['incerteza', 'polaridade', 'internacional']
    erros = []

    for r in resultados:
        idx = r.get('indice_original')
        if idx is None or idx >= len(df_noticias):
            continue

        for dim in dimensoes:
            valor_modelo = _codificar_valor(r.get(dim))
            if valor_modelo == "erro" or valor_modelo is None:
                continue

            if dim == 'internacional':
                valor_humano = 1 if str(df_noticias.loc[idx, dim]).strip().lower() == "sim" else 0
            else:
                valor_humano = df_noticias.loc[idx, dim]
                if isinstance(valor_humano, float) and str(valor_humano) == 'nan':
                    continue
                valor_humano = _codificar_valor(valor_humano)

            if valor_humano is None:
                continue

            try:
                if float(valor_modelo) != float(valor_humano):
                    erros.append({
                        'indice': idx,
                        'dimensao': dim,
                        'modelo': valor_modelo,
                        'humano': valor_humano,
                    })
            except (ValueError, TypeError):
                continue

    print(f"{'=' * 70}")
    print(f"MISCLASSIFICAÇÕES — {modulo}")
    print(f"{'=' * 70}")
    print()

    if not erros:
        print("Nenhuma misclassificação encontrada! O modelo acertou tudo.")
        return erros

    print(f"  {'Notícia':>8} {'Dimensão':<14} {'Modelo':>8} {'Humano':>8}")
    print(f"  {'-' * 8} {'-' * 14} {'-' * 8} {'-' * 8}")
    for e in erros:
        print(f"  #{e['indice']:<7d} {e['dimensao']:<14} {str(e['modelo']):>8} {str(e['humano']):>8}")
    print()
    print(f"Total: {len(erros)} divergências em {len(set(e['indice'] for e in erros))} notícias")
    print()
    print("Para ver os detalhes de uma notícia, use:")
    print(f'  ver_erro({erros[0]["indice"]}, modulo="{modulo}")')
    return erros


def ver_erro(indice, modulo="GPT-3.5-Turbo (regex)", truncar=2000):
    """
    Mostra os detalhes completos de uma notícia: texto, resposta do modelo,
    valores parseados, gabarito humano, e divergências.

    Parâmetros:
        indice (int): Índice da notícia no dataset (número da linha).
        modulo (str): Nome do módulo (ver listar_erros para opções).
        truncar (int ou None): Máximo de caracteres do texto da notícia.
            Use truncar=None para ver o texto completo.
    """
    _, resultados = _obter_resultados(modulo)
    if not resultados:
        print(f"Módulo '{modulo}' não foi executado.")
        return

    resultado = None
    for r in resultados:
        if r.get('indice_original') == indice:
            resultado = r
            break

    if resultado is None:
        print(f"Notícia #{indice} não encontrada nos resultados do módulo '{modulo}'.")
        return

    if indice >= len(df_noticias):
        print(f"Índice #{indice} fora do range do dataset (0-{len(df_noticias)-1}).")
        return

    noticia = df_noticias.loc[indice]
    dimensoes = ['incerteza', 'polaridade', 'internacional']

    sep = "=" * 70
    print(sep)
    print(f"ANÁLISE DETALHADA — Notícia #{indice}")
    print(f"Módulo: {modulo}")
    print(sep)

    # 1. Texto da notícia
    print()
    print("[ NOTÍCIA ]")
    print(f"  Jornal: {noticia.get('jornal', 'N/A')}")
    print(f"  Data:   {noticia.get('date', 'N/A')}")
    print()
    texto = str(noticia.get('noticia', ''))
    if truncar and len(texto) > truncar:
        print(f"  {texto[:truncar]}...")
        print(f"  (truncado em {truncar} chars — total: {len(texto)}. Use truncar=None para ver tudo)")
    else:
        print(f"  {texto}")

    # 2. Resposta bruta do modelo
    print()
    print("-" * 70)
    print("[ RESPOSTA BRUTA DO MODELO ]")
    raw = resultado.get('resposta_api_bruta', 'Não disponível')
    print(f"  {raw}")

    # 3. Comparação lado a lado
    print()
    print("-" * 70)
    print("[ COMPARAÇÃO: MODELO vs. HUMANO ]")
    print()
    print(f"  {'Dimensão':<16} {'Modelo':>10} {'Humano':>10} {'Resultado':>16}")
    print(f"  {'-' * 16} {'-' * 10} {'-' * 10} {'-' * 16}")

    divergencias = []
    for dim in dimensoes:
        val_modelo = _codificar_valor(resultado.get(dim))

        if dim == 'internacional':
            val_humano = 1 if str(noticia.get(dim, '')).strip().lower() == "sim" else 0
        else:
            val_humano = noticia.get(dim)
            if isinstance(val_humano, float) and str(val_humano) == 'nan':
                val_humano = None
            else:
                val_humano = _codificar_valor(val_humano)

        if val_modelo == "erro":
            status = "ERRO PARSING"
        elif val_humano is None:
            status = "sem gabarito"
        else:
            try:
                if float(val_modelo) == float(val_humano):
                    status = "OK"
                else:
                    status = ">>> DIVERGE <<<"
                    divergencias.append(dim)
            except (ValueError, TypeError):
                status = "incomparável"

        print(f"  {dim:<16} {str(val_modelo):>10} {str(val_humano):>10} {status:>16}")

    print()
    print("-" * 70)
    if divergencias:
        print(f"  RESULTADO: {len(divergencias)} divergência(s) em: {', '.join(divergencias)}")
    else:
        print("  RESULTADO: Modelo acertou todas as dimensões!")
    print(sep)


print("Funções definidas com sucesso:")
print("  - comparar_provedores()     — tabela de acurácia/precisão entre módulos")
print("  - listar_erros(modulo)      — lista misclassificações de um módulo")
print("  - ver_erro(indice, modulo)  — detalhes completos de uma notícia")


## 9.2 Comparação Automática

A célula abaixo detecta automaticamente quais módulos você executou e mostra a comparação lado a lado. Se você executou apenas o Módulo 1, verá só ele. Se executou os 3, verá os 3.

In [ ]:
# ============================================================
# 9.2 — Executando a comparação
# ============================================================

comparar_provedores()

## 9.3 Análise de Erros — Entendendo as Misclassificações

Agora vamos ver **onde** o modelo errou. Isso é fundamental para melhorar seu prompt!

**Como usar:**
1. A célula abaixo lista todas as misclassificações do Módulo 1 (baseline)
2. Escolha uma notícia da lista que te interesse
3. Use `ver_erro(NUMERO)` para ver todos os detalhes

> **Reflexão para o aluno:** Quando o modelo erra, será que o erro é do modelo ou da classificação humana? Em muitos casos, a classificação humana também é discutível — e essa é exatamente a conversa que queremos provocar!

In [ ]:
# ============================================================
# 9.3 — Listando misclassificações
# ============================================================

# Lista erros do Módulo 1 (baseline)
erros_modulo1 = listar_erros("GPT-3.5-Turbo (regex)")

# Se outros módulos foram executados, lista os erros deles também
if 'resumo_modulo2' in dir() and resumo_modulo2 is not None:
    print()
    erros_modulo2 = listar_erros("OpenAI Responses API")

if 'resumo_modulo3' in dir() and resumo_modulo3 is not None:
    print()
    erros_modulo3 = listar_erros("Google Gemini")

## 9.4 Inspecionando uma Notícia

Use a função `ver_erro()` para investigar uma notícia específica. Você verá:

1. **O texto completo da notícia** — para entender o contexto
2. **A resposta bruta do modelo** — exatamente o que o LLM retornou (antes do parsing)
3. **Comparação lado a lado** — valores do modelo vs. gabarito humano, com destaque nas divergências

**Exemplo:** troque o número abaixo por um índice da lista de erros acima.

In [ ]:
# ============================================================
# 9.4 — Inspeção detalhada de uma notícia
# ============================================================

# Troque o número pelo índice de uma notícia que te interessa!
# (os índices aparecem na lista de erros acima)

if erros_modulo1:
    # Exemplo: mostra a primeira notícia com erro
    primeiro_erro = erros_modulo1[0]['indice']
    ver_erro(primeiro_erro, modulo="GPT-3.5-Turbo (regex)")
else:
    print("Nenhum erro para inspecionar — o modelo acertou tudo!")

---

# 10. Desafios

Esta seção reúne os **desafios abertos** para voluntários. Cada desafio tem uma descrição, dicas para começar e critérios de sucesso. Escolha um (ou mais!) e documente suas descobertas.

> **Lembrete:** O objetivo não é apenas "fazer funcionar" — é **documentar o processo**. Anote o que tentou, o que deu errado, o que funcionou e por quê. Essa documentação é tão valiosa quanto o resultado final.

---

## Desafio 1: Fazer o Módulo 2 (OpenAI Moderna) Funcionar

**Dificuldade:** Média

**Objetivo:** Executar o pipeline de classificação usando a Responses API da OpenAI (Seção 7) e comparar os resultados com o Módulo 1.

### O que fazer

1. Vá à **Seção 7** e altere `EXECUTAR_MODULO_2 = True`
2. Execute a célula e observe os erros (se houver)
3. Use a tabela de troubleshooting (Seção 7, final) para diagnosticar
4. Corrija o código até que as 5 notícias sejam classificadas com sucesso
5. Compare as métricas com o Módulo 1

### Dicas

- Comece verificando se o SDK da OpenAI está atualizado: `pip install --upgrade openai`
- Se `client.responses.parse()` não existir, consulte a [documentação de migração](https://developers.openai.com/api/docs/guides/migrate-to-responses)
- O parâmetro `text_format` pode ter mudado de nome — consulte a documentação
- Use `gpt-4o-mini` (mais barato) para os testes iniciais

### Critério de sucesso

- [ ] As 5 notícias do teste rápido são classificadas sem erro
- [ ] Os resultados são convertidos para formato numérico
- [ ] As métricas são calculadas e comparadas com o Módulo 1
- [ ] Documentação: você anotou o que mudou e por quê

### Espaço para anotações

**O que eu precisei mudar para funcionar:**
- (descreva aqui)

**Erros encontrados e como resolvi:**
- (descreva aqui)

**Comparação de métricas com o Módulo 1:**
- (anote os valores aqui)

---

## Desafio 2: Fazer o Módulo 3 (Google Gemini) Funcionar

**Dificuldade:** Média

**Objetivo:** Executar o pipeline de classificação usando a API gratuita do Google Gemini (Seção 8) e comparar os resultados com os outros módulos.

### O que fazer

1. Certifique-se de que sua `GOOGLE_API_KEY` está configurada (Seção 3)
2. Vá à **Seção 8** e altere `EXECUTAR_MODULO_3 = True`
3. Execute a célula e observe os erros (se houver)
4. Use a tabela de troubleshooting (Seção 8, final) para diagnosticar
5. Compare as métricas com os Módulos 1 e 2

### Dicas

- Verifique se instalou o SDK correto: `google-genai` (NÃO `google-generativeai`)
- Se o modelo `gemini-2.5-flash` não funcionar, tente `gemini-2.0-flash`
- O `time.sleep(2)` entre chamadas respeita os limites do free tier — não remova
- Use a [documentação de Structured Output](https://ai.google.dev/gemini-api/docs/structured-output) como referência

### Critério de sucesso

- [ ] As 5 notícias do teste rápido são classificadas sem erro
- [ ] O JSON retornado é parseado corretamente
- [ ] As métricas são calculadas e comparadas com os Módulos 1 e 2
- [ ] Documentação: você anotou o que mudou e por quê

### Espaço para anotações

**O que eu precisei mudar para funcionar:**
- (descreva aqui)

**Erros encontrados e como resolvi:**
- (descreva aqui)

**Comparação de métricas com os outros módulos:**
- (anote os valores aqui)

---

## Desafio 3: Criar uma 5ª Dimensão de Classificação

**Dificuldade:** Alta

**Objetivo:** Adicionar uma nova dimensão de classificação ao pipeline e avaliar se o modelo consegue classificá-la corretamente.

### Ideias de novas dimensões

| Dimensão | Descrição | Valores possíveis |
|----------|-----------|-------------------|
| `setor` | Setor econômico principal da notícia | agropecuária, indústria, serviços, financeiro, governo |
| `urgencia` | Quão urgente/imediato é o impacto | alta, média, baixa |
| `fonte_incerteza` | De onde vem a incerteza | política fiscal, câmbio, inflação, emprego, geopolítica |
| `relevancia_br` | Relevância para a economia brasileira | alta, média, baixa |
| `horizonte` | Horizonte temporal do impacto | curto prazo, médio prazo, longo prazo |

### O que fazer

1. **Escolha** uma dimensão (da tabela acima ou invente a sua)
2. **Modifique o prompt** (Seção 6, `construir_prompt_v2`) para incluir a nova pergunta
3. **Modifique o regex** (ou use um dos módulos WIP com structured output) para extrair a nova dimensão
4. **Execute** a classificação com a nova dimensão
5. **Analise** os resultados qualitativamente (não haverá ground truth para a nova dimensão)

### Dicas

- Lembre-se: se usar o Módulo 1 (regex), você precisa criar um novo padrão regex para a nova dimensão
- Se usar os Módulos 2 ou 3 (structured output), basta adicionar um campo ao schema `ClassificacaoNoticia`
- Comece com valores simples (2-3 opções) antes de tentar categorias complexas
- Sem ground truth, avalie os resultados **qualitativamente** — leia as notícias e veja se as classificações fazem sentido

### Critério de sucesso

- [ ] Nova dimensão definida com valores claros
- [ ] Prompt modificado para incluir a nova pergunta
- [ ] Parsing adaptado (regex ou schema) para a nova dimensão
- [ ] Classificação executada com pelo menos 5 notícias
- [ ] Análise qualitativa: as classificações fazem sentido?
- [ ] Documentação: descrição da dimensão, decisões de design e resultados

### Espaço para anotações

**Dimensão escolhida:**
- Nome: (ex: `setor`)
- Valores possíveis: (ex: agropecuária, indústria, serviços)
- Por que escolhi essa dimensão: (justificativa)

**Modificações feitas:**
- Prompt: (o que mudou)
- Parsing: (regex novo ou campo no schema)

**Resultados:**
- (anote exemplos de classificações e se fazem sentido)

---

# 11. Recursos e Referências

Aqui estão os links e materiais de referência organizados por tópico. Consulte quando precisar de ajuda ou quiser se aprofundar.

---

## APIs de LLMs

### OpenAI

| Recurso | Link |
|---------|------|
| Documentação principal | [platform.openai.com/docs](https://platform.openai.com/docs) |
| Guia de Structured Outputs | [developers.openai.com/api/docs/guides/structured-outputs](https://developers.openai.com/api/docs/guides/structured-outputs) |
| Migração para Responses API | [developers.openai.com/api/docs/guides/migrate-to-responses](https://developers.openai.com/api/docs/guides/migrate-to-responses) |
| Modelos e deprecações | [developers.openai.com/api/docs/deprecations](https://developers.openai.com/api/docs/deprecations) |
| Status da API | [status.openai.com](https://status.openai.com/) |
| Guia de prompt engineering | [platform.openai.com/docs/guides/prompt-engineering](https://platform.openai.com/docs/guides/prompt-engineering) |

### Google Gemini

| Recurso | Link |
|---------|------|
| Documentação principal | [ai.google.dev/gemini-api/docs](https://ai.google.dev/gemini-api/docs) |
| Quickstart | [ai.google.dev/gemini-api/docs/quickstart](https://ai.google.dev/gemini-api/docs/quickstart) |
| Structured Output | [ai.google.dev/gemini-api/docs/structured-output](https://ai.google.dev/gemini-api/docs/structured-output) |
| Modelos disponíveis | [ai.google.dev/gemini-api/docs/models](https://ai.google.dev/gemini-api/docs/models) |
| Preços e limites | [ai.google.dev/gemini-api/docs/pricing](https://ai.google.dev/gemini-api/docs/pricing) |
| Criar chave de API | [aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey) |
| SDK Python (google-genai) | [googleapis.github.io/python-genai](https://googleapis.github.io/python-genai/) |

---

## Bibliotecas Python

| Biblioteca | Para que usamos | Documentação |
|-----------|----------------|--------------|
| **pandas** | Manipulação de DataFrames | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **scikit-learn** | Métricas de classificação | [scikit-learn.org](https://scikit-learn.org/stable/modules/model_evaluation.html) |
| **Pydantic** | Schemas de validação | [docs.pydantic.dev](https://docs.pydantic.dev/) |
| **openai** (SDK) | Cliente da API OpenAI | [github.com/openai/openai-python](https://github.com/openai/openai-python) |
| **google-genai** (SDK) | Cliente da API Gemini | [googleapis.github.io/python-genai](https://googleapis.github.io/python-genai/) |

---

## Prompt Engineering

| Recurso | Descrição |
|---------|-----------|
| [OpenAI Prompt Engineering Guide](https://platform.openai.com/docs/guides/prompt-engineering) | Guia oficial da OpenAI com técnicas práticas |
| [Prompting Guide](https://www.promptingguide.ai/) | Guia comunitário abrangente sobre técnicas de prompting |
| [Learn Prompting](https://learnprompting.org/) | Curso interativo gratuito sobre prompt engineering |

---

## Sobre o Projeto IBRE/FGV

| Recurso | Descrição |
|---------|-----------|
| [IBRE/FGV](https://ibre.fgv.br/) | Site oficial do Instituto Brasileiro de Economia |
| [Indicador de Incerteza da Economia (IIE-Br)](https://portalibre.fgv.br/indicador-de-incerteza-da-economia) | Página do indicador que motivou este projeto |

---

**Fim do notebook!** Esperamos que esta experiência tenha sido útil para entender como LLMs podem ser aplicados a tarefas de classificação de texto. Bom trabalho!